# Plug-and-Play (PnP) — Homework Notebook

Welcome! This notebook is your homework on the **Plug-and-Play (PnP)** framework for solving inverse problems in imaging. It mirrors the 12-section lecture notes you have been studying.

## How this homework is organized

For each of the 12 sections you will find:
1. A short **theory recap** in a markdown cell (read it carefully — it is the only theory you get).
2. A **code cell with TODOs** that you must complete yourself.
3. A **Questions** markdown cell where you write your answers in your own words.

## Rules

- **Do not delete the TODO comments.** They are your spec — replace `None`, `raise NotImplementedError`, and `# TODO:` lines with real code.
- Make the notebook **fully runnable top to bottom** before submitting. We will run `Restart Kernel & Run All` on your file.
- Use **only** the libraries imported at the top of each section unless a TODO explicitly asks for something else.
- Reuse variables across sections where the notes do (e.g. `img`, `y`, `H`, `psf`). The cells assume those names exist from Section 1 / 4.
- For every Questions cell: write at least **2–4 sentences per question** in your own words. Yes/No answers will not be graded.

## Grading

- Code TODOs: **60%**
- Question answers: **30%**
- Notebook runs cleanly + plots look right: **10%**

Good luck — and remember that every TODO is solvable with code you have already seen in the lecture notes.


## 1. Introduction: Inverse Problems

Many tasks in imaging and data science (denoising, deblurring, CT reconstruction, MRI, …) are **inverse problems**.

* **Forward problem:** we know inputs and the process, we compute outputs. Easy.
* **Inverse problem:** we observe the output and we want to recover the input. Usually **ill-posed** (many possible inputs give the same observation).

In images:

* $x$ = the unknown clean image we want.
* $A$ = the *forward model* (e.g. blur, downsampling, projection, …).
* $y$ = the observed data, usually $y = A x + \text{noise}$.

We pick the most plausible $x$ by minimizing a cost with two terms:

$$\hat{x} = \arg\min_x \left\{ f(x) + h(x) \right\}$$

* $f(x)$ — **data-fidelity**: how well $A x$ matches the observation $y$. Typical choice for Gaussian noise: $f(x) = \tfrac{1}{2\sigma^2}\|A x - y\|_2^2$.
* $h(x)$ — **regularizer / prior**: how "natural" $x$ looks (smoothness, sparsity, learned prior, …).

The two terms fight each other. The minimizer is a compromise.

### What you will do in this section

1. Build a clean image $x$.
2. Define a simple, known forward model $A$ (a horizontal motion blur).
3. Generate observed data $y = A x + n$ with Gaussian noise.
4. Implement $f(x)$ and check it on three scenarios: a perfect guess vs. clean data, a perfect guess vs. noisy data, and a bad guess vs. noisy data.
5. Plot $x$ and $y$.


In [ ]:
# =============================================================
# Section 1 — Inverse Problems: forward model, data, and f(x)
# =============================================================
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color, img_as_float
from scipy.signal import convolve2d

# --- 1. Load and prepare the clean image x ---------------------------------
# TODO: Load the skimage 'astronaut' image, convert it to grayscale, and cast
#       it to float (so pixel values live in [0, 1]).

img = img_as_float(color.rgb2gray(data.astronaut()))[64:320, 64:320]

# TODO: Crop a square 256x256 region of your choice so later cells run fast.
# Store the result in a variable called `img`.

#Already done that

# --- 2. Define the forward model A ----------------------------------------
# We will use a simple horizontal motion blur of length 9.
# TODO: Build `psf`, a 9x9 numpy array that:
#         * is zero everywhere EXCEPT the middle row,
#         * has the middle row filled with 1/9 (so the kernel sums to 1).
psf = np.zeros((9, 9))
psf[4, :] = 1.0 / 9.0


# TODO: Define a function (or lambda) `A` that applies psf to an image.
#       Use scipy.signal.convolve2d with mode='same' and boundary='wrap'.
A = lambda x: convolve2d(x, psf, mode='same', boundary='wrap')

# --- 3. Create the observed data y = A(x) + noise --------------------------
# TODO: Pick a noise standard deviation. Start with sigma = 0.01.
sigma = 0.01

# TODO: Compute y_clean = A(img), then add Gaussian noise of std `sigma` to
#       obtain y. Use np.random.randn for the noise.
y_clean = A(img)
y = y_clean + sigma * np.random.randn(*y_clean.shape)

# --- 4. Implement the data-fidelity f(x) ----------------------------------
def data_fidelity(x, y, sigma=sigma):
    """L2 data-fidelity:  f(x) = (1 / (2 * sigma^2)) * || A(x) - y ||_2^2.

    TODO: implement this. You should use the forward model `A` defined above
    and a squared L2 norm (sum of squared differences). Return a single
    scalar number.
    """
    return (1.0 / (2.0 * sigma**2)) * np.sum((A(x) - y)**2)

# --- 5. Sanity-check f on three scenarios ---------------------------------
# Compute and print THREE costs (each one a single number):
#   (a) cost_perfect        = f(img, y_clean)              -> ~ 0
#   (b) cost_best_realistic = f(img, y)                    -> non-zero, due to noise
#   (c) cost_bad_guess      = f(zeros_like(img), y)        -> a much larger number
#
# TODO: Compute (a), (b), (c) and print them with descriptive labels so the
#       output is easy to read.
cost_perfect = data_fidelity(img, y_clean)
cost_best_realistic = data_fidelity(img, y)
cost_bad_guess = data_fidelity(np.zeros_like(img), y)

print(f"Cost perfect (clean data): {cost_perfect:.4f}")
print(f"Cost best realistic (noisy data): {cost_best_realistic:.4f}")
print(f"Cost bad guess (noisy data): {cost_bad_guess:.4f}")

# TODO (mini-question, write your prediction *as a comment* before running):
#       Roughly how many times larger do you expect cost_bad_guess to be
#       compared to cost_best_realistic?
# I suppose that cost_bad_guess is much larger than cost_best_realistic.
# y = A(img) + noise, so f(img, y) = ||noise||^2 / (2σ^2) ≈ N/2 (where N = 256*256).
# f(zeros, y) = ||A(img)||^2 / (2σ^2), which is approx. (mean pixel value)^2 * N / (2σ^2).
# Rough ratio: (mean_pixel / sigma)^2 ≈ (0.5 / 0.01)^2 = 2500x larger.
print(f"    ratio (c)/(b)         = {cost_bad_guess / cost_best_realistic:.1f}")

# --- 6. Show the original image and the measurement -----------------------
# TODO: Use plt.subplots(1, 2, figsize=(8, 4)) to display `img` and `y` side
#       by side in grayscale with informative titles. Hide the axes.

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img, cmap='gray')
ax[0].set_title('Clean Image (x)')
ax[0].axis('off')

ax[1].imshow(y, cmap='gray')
ax[1].set_title('Observed Data (y)')
ax[1].axis('off')
plt.show()

### Section 1 — Questions

Answer in this markdown cell (double-click to edit). Write **2–4 sentences per question**.

1. The "Perfect Guess vs. Clean Data" cost is exactly 0. Why? Plug $x = \text{img}$ into the formula for $f$ and explain.
2. The "Perfect Guess vs. Noisy Data" cost is *not* zero. Express the **expected value** of this cost as a function of $\sigma$ and the number of pixels $N$. (Hint: the noise is i.i.d. Gaussian and $f$ already has a $1/(2\sigma^2)$ in front.)
3. If you double $\sigma$ when generating $y$, what happens to (a) the look of $y$, and (b) the cost of the perfect guess on the new $y$? Justify.
4. Our forward model $A$ is a 9-tap horizontal averaging filter. Is the inverse problem **ill-posed**? Explain with one concrete example of two different images $x_1 \neq x_2$ that would give the same $A x$.

*Your answers:*

**1.** When x=img, we have A(x)=y_clean, so the residual A(x)−y_clean=0 identically. The squared norm of the zero vector is zero, making f=0 regardless of σ.

---
**2.** The residual at the perfect guess is:

$$Ax - y = y_\text{clean} - (y_\text{clean} + n) = -n, \quad n \sim \mathcal{N}(0, \sigma^2 I)$$

So the cost becomes:

$$f(\text{img}, y) = \frac{1}{2\sigma^2}\|n\|^2 = \frac{1}{2\sigma^2}\sum_{i=1}^{N} n_i^2$$

Taking the expectation, since $\mathbb{E}[n_i^2] = \sigma^2$:

$$\mathbb{E}\left[f(\text{img}, y)\right] = \frac{1}{2\sigma^2} \cdot N\sigma^2 = \frac{N}{2}$$

The expected cost is $\dfrac{N}{2}$, independent of $\sigma$.

---
**3.** $\textbf{(a)}$ The observed data $y$ becomes visibly noisier. Since $y = Ax + n$ and $\sigma \to 2\sigma$,
the noise standard deviation doubles:

$$\text{Var}(y_i) = \sigma^2 \to 4\sigma^2$$


$\textbf{(b)}$ The expected cost of the perfect guess remains unchanged. With $\sigma \to 2\sigma$:
$$
\mathbb{E}\left[f(\text{img}, y)\right]
= \frac{1}{2(2\sigma)^2} \cdot N(2\sigma)^2
= \frac{N \cdot 4\sigma^2}{2 \cdot 4\sigma^2}
= \frac{N}{2}
$$

The $\sigma^2$ terms cancel, leaving the expected cost at $\dfrac{N}{2}$ regardless of $\sigma$.

---
**4.** The problem is ill-posed because the forward operator $A$ is not injective.

Define the null space of $A$:

$$\mathcal{N}(A) = \{v : Av = 0\}$$

For a 9-tap horizontal averaging filter, any signal $v$ that is periodic with period 9
along rows and sums to zero over each window satisfies $Av = 0$.
Then for any such $v \neq 0$:

$$A(x + v) = Ax + Av = Ax = y$$

So $x$ and $x + v$ produce identical measurements, making recovery of $x$ from $y$ alone
impossible without additional constraints.


## 2. What is "Plug-and-Play"?

We saw:

$$\hat{x} = \arg\min_x \{ f(x) + h(x) \}$$

Iterative algorithms (proximal methods, ADMM, …) solve this by alternating between:

1. A step that lowers $f(x)$ — "match the data".
2. A step that lowers $h(x)$ — "look natural".

**Plug-and-Play (PnP)** focuses entirely on step 2.

### The proximal map of $h$

Step 2 is typically the **proximal map** of $h$:

$$\mathrm{prox}_{\lambda h}(v) \;=\; \arg\min_x \left\{ \tfrac{1}{2}\|x-v\|_2^2 + \lambda h(x) \right\}.$$

Read it carefully:

* $\|x-v\|_2^2$ → "stay close to the noisy input $v$".
* $\lambda h(x)$ → "look natural under our prior".

That is **exactly the job description of a denoiser**: take a noisy input, return something close to it that looks clean.

### The PnP trick

Solving for $\mathrm{prox}_{\lambda h}$ analytically is painful for complicated priors $h$. PnP says: **forget the formula, just call a denoiser**.

* Traditional step: $x \leftarrow \mathrm{prox}_{\lambda h}(v)$ — hard math.
* PnP step:        $x \leftarrow D_{\sigma}(v)$         — call BM3D / NLM / a CNN / a diffusion model.

We will visualize this idea by treating Non-Local Means (NLM) as a black-box denoiser.


In [ ]:
# =============================================================
# Section 2 — Treating a denoiser as the proximal map of h
# =============================================================
# We will use scikit-image's Non-Local Means (NLM) as our "black box"
# denoiser. The clean image `img` is the one you built in Section 1.
from skimage.restoration import denoise_nl_means

# --- 1. Build a noisy version of `img` ------------------------------------
# We want a noticeably noisy image so the denoiser's effect is visible.
# TODO: Pick a noise std (try 0.05) and store it in `noise_std_sec2`.
noise_std_sec2 = 0.05

# TODO: Add Gaussian noise of std `noise_std_sec2` to `img`. Store the
#       result in `noisy`. Clip the result to [0, 1] so it is a valid image.
noisy = np.clip(img + noise_std_sec2 * np.random.randn(*img.shape), 0, 1)

# --- 2. Apply the NLM denoiser --------------------------------------------
# TODO: Call denoise_nl_means(...) on `noisy` and store the output in
#       `denoised`. Useful kwargs to know (from the docs):
#         patch_size      : small patch comparison window (e.g. 5)
#         patch_distance  : max search distance for similar patches (e.g. 6)
#         h               : smoothing strength (try 0.08; higher = smoother)
#         fast_mode       : True  (faster, approximate)
#       Pick reasonable values yourself — do NOT just copy the lecture's
#       numbers without trying others first.
denoised = denoise_nl_means(noisy, patch_size=5, patch_distance=6, h=0.04, fast_mode=True)

# --- 3. Visualize ----------------------------------------------------------
# TODO: Plot a 1x3 row of grayscale images: original `img`, `noisy`, and
#       `denoised`. Add titles. Hide axes.

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img, cmap='gray')
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(noisy, cmap='gray')
ax[1].set_title('Noisy Image')
ax[1].axis('off')

ax[2].imshow(denoised, cmap='gray')
ax[2].set_title('Denoised (NLM)')
ax[2].axis('off')
plt.show()

### Section 2 — Questions

1. Look at the proximal map formula again: $\mathrm{prox}_{\lambda h}(v) = \arg\min_x \{\tfrac{1}{2}\|x - v\|_2^2 + \lambda h(x)\}$. In one or two sentences each, explain in plain English what the **two terms** of the objective ask the solution $x$ to do, and why that matches what a denoiser does.
2. Try setting `h=0.01`, then `h=0.3` in your NLM call (re-run the cell each time). Describe in your own words how the output changes. Which one is closer to the original `img` for **your** noise level? Which one might be useful inside an iterative loop where the input is *very* noisy?
3. PnP "replaces" $\mathrm{prox}_{\lambda h}$ with any off-the-shelf denoiser $D_\sigma$. Give one **practical** advantage and one **theoretical** worry of doing this swap.
4. Suppose your denoiser is the identity, $D(v) = v$ (it does nothing). What would the implicit prior $h$ look like? Is this a "strong" or "weak" prior?

*Your answers:*

**1.** The first term $\frac{1}{2}\|x - v\|^2$ asks $x$ to stay close to the
input $v$ (fidelity to the noisy observation). The second term
$\lambda h(x)$ asks $x$ to be ``natural'' or regular according to the
prior $h$. A denoiser does exactly this trade-off: it nudges the noisy
input toward the manifold of clean images without straying too far from
the measurement.

---
**2.** With $\texttt{h=0.01}$ the filter is very conservative---the output looks
almost identical to the noisy input because the smoothing weight is too
small to suppress the noise. With $\texttt{h=0.3}$ the filter
over-smooths: fine details and edges are blurred away, giving a
painterly, flat result. For the noise level $\sigma=0.05$ used here,
$\texttt{h=0.04}$ is closest to the original $\texttt{img}$ and also denoised.
A large value such as $\texttt{h=0.3}$ would be useful inside an
iterative loop where the intermediate iterate is very noisy, because
aggressive denoising pulls the solution quickly toward the clean-image
manifold.

---
**3.** Practical advantage: Any state-of-the-art denoiser
(NLM, BM3D, a deep network) can be dropped in without deriving a
closed-form proximal operator, giving immediate access to powerful,
hand-tuned or learned priors.

Theoretical worry: The substituted denoiser may not correspond
to the gradient or proximal operator of any proper convex
function $h$, so classical convergence guarantees of the underlying
algorithm (e.g.\ ADMM, ISTA) no longer apply and the iteration may
diverge or cycle.

---
**4.** The proximal map of $h$ at $v$ is
$\mathrm{prox}_{\lambda h}(v) = \arg\min_x
\bigl\{\tfrac{1}{2}\|x-v\|^2 + \lambda h(x)\bigr\}$.
If $D(v)=v$ for all $v$, the minimiser is always $x=v$, which requires
$\nabla h(x)=0$ everywhere---i.e.\ $h$ is a constant (flat) function.
A flat prior assigns equal probability to every image, so it is the
weakest possible prior: it encodes no preference for smooth,
sparse, or otherwise structured images whatsoever.


## 3. Modeling the Forward Problem (how $x$ becomes $y$)

The data-fidelity $f(x)$ is determined by the **forward model** $A$. There are three common families:

### (a) Traditional physics model
You know the physics well enough to write $A$ as an explicit formula.

* Examples: a known blur kernel (our `psf`), the Radon transform (CT), an MRI subsampling mask.
* You get a clean $f(x) = \tfrac{1}{2\sigma^2}\|A x - y\|_2^2$ that you can differentiate or invert in the Fourier domain.

### (b) Action / operational model (black box)
You don't have a clean formula, but you have a *device* or *program* that maps $x \to y$.

* Examples: a real camera pipeline with proprietary denoising / tonemapping / JPEG compression.
* You can *call* $A$ but can't easily differentiate it or invert it.

### (c) Synthetic / virtual model (for training data)
You build a **simulator** that mimics the real forward process, and use it to generate synthetic $(x, y)$ pairs to train a learning-based reconstructor.

* The danger: **domain gap** — synthetic $y$ may not look like real $y$.

### Where PnP fits

PnP decouples the prior $h$ from the data term $f$. If $A$ is messy (case b), PnP lets you keep using a strong, generic denoiser as $h$ without redesigning everything.

In *this* notebook our $A$ is a **traditional physics model** (a known convolution).


In [ ]:
# =============================================================
# Section 3 — Inspecting our traditional physics model
# =============================================================
# The `psf` kernel you defined in Section 1 IS our forward model.
# Since it's not a black box, we can inspect its properties.

# TODO: Print a short report about `psf` that includes at least:
#         * a one-line label saying which type of forward model it is
#           (traditional / action / synthetic),
#         * `psf.shape`,
#         * `np.sum(psf)`  (it should be 1 — explain why in the questions).

print("Forward model classification: Traditional physics model")
print(f"psf.shape: {psf.shape}")
print(f"np.sum(psf): {np.sum(psf)}")

# TODO: Visualize `psf` itself as a small image with plt.imshow.
#       Use a colormap that makes the bright row obvious, add a colorbar,
#       and a title.

plt.figure(figsize=(4, 4))
plt.imshow(psf, cmap='hot', interpolation='nearest')
plt.colorbar()
plt.title("PSF (9-tap Horizontal Blur)")
plt.show()

### Section 3 — Questions

1. Why must our blur kernel `psf` sum to **1**? What would change in the observed image $y$ if `sum(psf) = 0.5` instead? What if `sum(psf) = 2`?
2. Classify each of the following forward models as **traditional**, **action**, or **synthetic**:
   * (a) A Radon transform used in CT reconstruction.
   * (b) A trained generative model that produces realistic ultrasound images from anatomical labels (used to make training pairs).
   * (c) The entire image processing pipeline inside an iPhone camera (RAW $\to$ JPEG).
   * (d) A Fourier subsampling mask on k-space for compressed-sensing MRI.
3. Suppose you are working with case (c) above (a real camera pipeline, fully black box). Explain in 2–3 sentences why PnP is a particularly attractive framework here, compared to a method that needs the gradient of $A$.
4. For our `psf`, the kernel is **separable** (it only smears horizontally). What does this tell you about features that align with the *vertical* axis vs. the *horizontal* axis after blurring? Which features will be hardest to recover?

*Your answers:*

**1.** The PSF computes a weighted average of neighboring pixels. Summing to 1 ensures
energy (brightness) is conserved: a uniform image stays uniform after blurring.

For a constant image $x = c$:
$$
y = A(x) = \left(\sum_k \mathrm{psf}_k\right) \cdot c
$$
So $\sum \mathrm{psf} = 1$ is the flux-conservation condition.


-$\sum \mathrm{psf} = 0.5$: the observed image $y$ is darker by a factor of 2.

-$\sum \mathrm{psf} = 2$: the observed image $y$ is brighter by a factor of 2,likely saturating (clipping to 1).

---
**2.**

(a) Radon transform $\to$ $\textbf{Traditional}$ (explicit analytical formula).

(b) Generative model producing synthetic pairs $\to$ $\textbf{Synthetic}$.

(c) iPhone RAW$\to$JPEG pipeline $\to$ $\textbf{Action}$ (black-box, callable but not differentiable).

(d) Fourier subsampling mask $\to$ $\textbf{Traditional}$ (explicit linear operator in $k$-space).

---
**3.** Gradient-based methods require $\nabla_x f(x) = \nabla_x \tfrac{1}{2\sigma^2}\|A(x)-y\|_2^2$,
which demands the Jacobian of $A$ that is infeasible for a closed camera pipeline.
PnP decouples the data-fidelity step from the prior step: it only requires
evaluating $A(x)$ to compute the residual, while the denoiser $D_\sigma$ handles
the prior entirely separately. This makes PnP directly applicable to black-box
forward models without any modification.

---
**4.** The PSF averages along horizontal rows only, so:

-Vertical features (edges aligned with the vertical axis) are unaffected — vertical structure is preserved across rows.

-Horizontal features (fine detail along the blur direction) are smeared over 9 pixels.

The hardest features to recover are fine horizontal details that are high-frequency
content along the blur direction. In the Fourier domain, $A$ attenuates or zeroes
out high horizontal frequencies, making them unrecoverable without a strong prior.


## 4. ADMM — splitting the problem

**Alternating Direction Method of Multipliers (ADMM)** solves

$$\min_x \; f(x) + h(x)$$

by introducing an auxiliary variable $v$ and forcing $x = v$:

$$\min_{x, v} \; f(x) + h(v) \quad \text{s.t.} \quad x - v = 0.$$

That trick lets us *separate* $f$ and $h$. Using a scaled dual variable $u$, ADMM iterates **three updates**:

1. **$x$-update:** $\;\; x^{k+1} = \arg\min_x \; f(x) + \tfrac{\rho}{2}\|x - v^k + u^k\|_2^2$
2. **$v$-update:** $\;\; v^{k+1} = \arg\min_v \; h(v) + \tfrac{\rho}{2}\|x^{k+1} - v + u^k\|_2^2$
3. **$u$-update:** $\;\; u^{k+1} = u^k + x^{k+1} - v^{k+1}$

### PnP step

Look at the $v$-update — it's *exactly* the proximal map of $h$ applied to $(x^{k+1} + u^k)$. PnP **replaces** it by a denoiser:

$$v^{k+1} = D_\sigma(x^{k+1} + u^k)$$

### Solving the $x$-update with the FFT

For our problem $f(x) = \tfrac{1}{2}\|A x - y\|_2^2$, set $z^k = v^k - u^k$. Setting the gradient of the $x$-update to zero gives the **normal equation**:

$$(A^\top A + \rho I)\, x = A^\top y + \rho z^k.$$

When $A$ is a circular convolution with kernel `psf`, this is **diagonal in the Fourier domain**. Let $H = \mathrm{FFT}(\text{padded psf})$. Then

$$X = \frac{\overline{H}\, Y + \rho\, Z}{|H|^2 + \rho}$$

and $x = \mathrm{Re}(\mathrm{IFFT}(X))$. This is *one matrix-free step*.

### What you will implement

* A helper `fft_psf(psf, shape)` that returns the FFT of a zero-padded kernel.
* The function `x_update_fft(y, psf_fft, rho, z)` that solves the normal equation in the Fourier domain in a single shot.


In [ ]:
# =============================================================
# Section 4 — FFT-based x-update for the ADMM loop
# =============================================================
import numpy as np
from numpy.fft import fft2, ifft2

# --- 1. Pad the PSF and take its FFT --------------------------------------
def fft_psf(psf, shape):
    """Return the FFT of `psf` zero-padded up to `shape`.

    Steps you must implement:
      * Create an all-zero array of the target `shape`.
      * Copy `psf` into its top-left corner.
      * Return fft2 of that array.
    """
    # TODO: implement the three steps above.
    padded_psf = np.zeros(shape)
    padded_psf[:psf.shape[0], :psf.shape[1]] = psf
    return fft2(padded_psf)

# Once `psf` (from Section 1) and `img` exist, pre-compute H = FFT(padded psf).
# TODO: Compute H = fft_psf(psf, img.shape) and bind it to the name `H`.
H = fft_psf(psf, img.shape)

# --- 2. The x-update in the Fourier domain --------------------------------
def x_update_fft(y, psf_fft, rho, z):
    """Solve  (A^T A + rho I) x = A^T y + rho z  for x, using the FFT.

    For a circular convolution A with OTF H = psf_fft, the solution is
        X = ( conj(H) * Y + rho * Z ) / ( |H|^2 + rho )
    and x = real(ifft2(X)).

    Parameters
    ----------
    y       : 2D numpy array  -- the observed (blurry/noisy) image
    psf_fft : 2D complex array -- output of fft_psf(psf, y.shape)
    rho     : float            -- ADMM penalty parameter
    z       : 2D numpy array   -- z = v - u from the previous ADMM step

    Returns
    -------
    x : 2D numpy array -- the result of one x-update
    """
    # TODO: implement, following the equation in the docstring.
    # Suggested intermediate variables:
    #   Hc        = np.conj(psf_fft)
    #   Y, Z      = FFT of y and z
    #   numerator   = Hc * Y + rho * Z
    #   denominator = Hc * psf_fft + rho            (this is |H|^2 + rho)
    #   X = numerator / denominator
    #   x = np.real(ifft2(X))
    Hc = np.conj(psf_fft)
    Y = fft2(y)
    Z = fft2(z)
    
    numerator = Hc * Y + rho * Z
    denominator = Hc * psf_fft + rho 
    
    X = numerator / denominator
    return np.real(ifft2(X))

# --- 3. Quick sanity check ------------------------------------------------
# TODO: Run a one-shot test of your function using `y` (from Section 1) and a
#       placeholder z = img.copy(). Use rho = 0.02. Print the output shape
#       (must equal img.shape) and the min/max of the result. This is NOT
#       the final reconstruction yet — we only check the function runs.

x_test = x_update_fft(y, H, 0.02, img.copy())
print(f"Output shape: {x_test.shape} (Must match {img.shape})")
print(f"Min value: {x_test.min():.4f}, Max value: {x_test.max():.4f}")


### Section 4 — Questions

1. Starting from $\arg\min_x \tfrac{1}{2}\|A x - y\|_2^2 + \tfrac{\rho}{2}\|x - z\|_2^2$, set the gradient w.r.t. $x$ to zero and **derive** the normal equation $(A^\top A + \rho I)\, x = A^\top y + \rho z$ in your own writing. Show every step.
2. When $A$ is a circular convolution with kernel `psf`, why is the system $(A^\top A + \rho I) x = \ldots$ **diagonal** in the Fourier basis? Refer to the convolution theorem in your answer.
3. What is the role of the parameter $\rho$ in the $x$-update?
   * What happens if $\rho \to 0$? Look at the formula $X = \frac{\overline{H} Y + \rho Z}{|H|^2 + \rho}$.
   * What happens if $\rho \to \infty$?
   * Which regime gives more weight to the data, and which gives more weight to $z$ (the previous denoiser output)?
4. We pad `psf` into the top-left corner of a zero array. Why is **circular** (a.k.a. "wrap") boundary handling consistent with this padding? What artifact would you expect if you used `boundary='symm'` in `convolve2d` but kept the same FFT setup?

*Your answers:*

**1.**
Define the objective:

$$F(x) = \frac{1}{2}\|Ax - y\|_2^2 + \frac{\rho}{2}\|x - z\|_2^2$$

Expand each term:

$$F(x) = \frac{1}{2}(Ax - y)^\top(Ax - y) + \frac{\rho}{2}(x - z)^\top(x - z)$$

Take the gradient with respect to $x$:

$$\nabla_x F = \frac{1}{2} \cdot 2A^\top(Ax - y) + \frac{\rho}{2} \cdot 2(x - z) = A^\top A x - A^\top y + \rho x - \rho z$$


Set $\nabla_x F = 0$:

$$A^\top A x + \rho x = A^\top y + \rho z$$

$$\boxed{(A^\top A + \rho I)\, x = A^\top y + \rho z}$$

---

**2.**
The convolution theorem states that circular convolution in the spatial domain
corresponds to pointwise multiplication in the Fourier domain:

$$\mathcal{F}\{Ax\} = H \cdot X,$$

where $H = \mathcal{F}\{\mathrm{psf}\}$ and $X = \mathcal{F}\{x\}$.
This means $A$ is diagonalized by the DFT matrix $\mathcal{F}$:

$$A = \mathcal{F}^{-1} \operatorname{diag}(H)\, \mathcal{F}.$$

Therefore:
$$
A^\top A
  = \mathcal{F}^{-1} \operatorname{diag}(\bar{H})\, \mathcal{F}
    \cdot
    \mathcal{F}^{-1} \operatorname{diag}(H)\, \mathcal{F}
  = \mathcal{F}^{-1} \operatorname{diag}(|H|^2)\, \mathcal{F}.
$$

Adding $\rho I$ (diagonal in any basis):
$$
A^\top A + \rho I
  = \mathcal{F}^{-1} \operatorname{diag}(|H|^2 + \rho)\, \mathcal{F}.
$$

So in the Fourier basis the system is diagonal.

---
**3.**
Starting from:

$$X = \frac{\bar{H}\, Y + \rho\, Z}{|H|^2 + \rho}$$

As $\rho \to 0$:

$$X \approx \frac{\bar{H}\, Y}{|H|^2} = \frac{Y}{H}.$$

This is the naive deconvolution (pseudo-inverse of $A$). The data term dominates
completely. At frequencies where $|H_k| \approx 0$, the solution blows up and the result is unstable and extremely noisy.

As $\rho \to \infty$:

$$X \approx \frac{\rho\, Z}{\rho} = Z.$$

The $x$-update simply returns $z$, the previous denoiser output.
The data fidelity is ignored entirely.


$\rho$ controls the coupling strength between the two ADMM subproblems.
In practice it is tuned so that neither term overwhelms the other across iterations.

---
**4.**
Placing the PSF at the top-left corner aligns with the DFT’s implicit periodicity assumption, making the FFT compute exact circular convolution. This is consistent with boundary='wrap', so the forward model and the x -update invert the same operator. Using boundary='symm' instead would create a mismatch: 

y is generated with symmetric extension, but the solver inverts a circular-convolution model, causing boundary artifacts (ringing/blurring) near the image edges.


## 5. Denoisers — the "Plug" in Plug-and-Play

In PnP-ADMM the $v$-update is

$$v^{k+1} = D_\sigma(x^{k+1} + u^k).$$

The denoiser $D_\sigma$ plays the role of the prior $h$. We never write down a formula for $h$.

Two families of denoisers are common:

### Classical algorithms

* **NLM (Non-Local Means)** — assumes images contain repeating patches. Each pixel is replaced by a weighted average of similar patches elsewhere in the image.
* **BM3D (Block-Matching and 3D filtering)** — groups similar patches into a 3D stack, filters the stack jointly in a transform domain, and reassembles. State-of-the-art classical denoiser.

### Learned denoisers (CNNs)

* **DnCNN** — a small ResNet-style CNN with residual learning (predicts the *noise*, then subtracts).
* **U-Net, denoising diffusion priors** — heavier, more expressive denoisers.

> Note: a strong denoiser need not be the *true* proximal map of any explicit $h$. The empirical PnP literature shows it works anyway.

### What you will do

1. Add noise to `img`.
2. Run NLM, BM3D, and an **untrained** TinyDnCNN on it.
3. Compare what they produce. The untrained CNN should look terrible — that's the point.


In [ ]:
# Install BM3D (only needed once per environment). Comment this out if it
# is already installed, or if installation fails on your machine.
# TODO: leave this cell as-is, just run it once. If BM3D refuses to install,
#       continue without it — the next cell tolerates a missing BM3D.
%pip install bm3d


In [ ]:
# =============================================================
# Section 5 — Three denoisers as black-box "plugs"
# =============================================================
import numpy as np
import matplotlib.pyplot as plt
from skimage.restoration import denoise_nl_means
import torch
import torch.nn as nn

# --- 1. Build a noisy version of `img` ------------------------------------
# TODO: Pick a noise std (suggest 0.06), save it in `noise_std_sec5`, and
#       build `noisy_img` = img + Gaussian noise, clipped to [0, 1].
noise_std_sec5 = 0.06
noisy_img = np.clip(img + noise_std_sec5 * np.random.randn(*img.shape), 0, 1)

# --- 2. Classical denoiser: NLM -------------------------------------------
# TODO: Run denoise_nl_means on `noisy_img.astype(np.float64)` with reasonable
#       hyperparameters. Store the result in `denoised_nlm`.
denoised_nlm = denoise_nl_means(noisy_img.astype(np.float64), patch_size=5,patch_distance=6, h=0.08, fast_mode=True)

# --- 3. Classical denoiser: BM3D ------------------------------------------
# BM3D is a separate package; guard the import so the cell still runs without it.
denoised_bm3d = None
bm3d_available = False
try:
    from bm3d import bm3d
    # TODO: Call bm3d(...) on `noisy_img`. Pass the known noise std via the
    #       keyword `sigma_psd=...`. Store the result in `denoised_bm3d`.
    denoised_bm3d = bm3d(noisy_img, sigma_psd=noise_std_sec5)
    bm3d_available = True
except Exception as e:
    print(f"BM3D not available, skipping. Error: {e}")

# --- 4. A tiny (untrained) DnCNN ------------------------------------------
# IMPORTANT: this network has RANDOM weights. We are only testing the
# *architecture* and the data plumbing. It will NOT denoise well.
class TinyDnCNN(nn.Module):
    """A small residual-learning CNN. It predicts the noise; we subtract it
    from the input to get the clean image.

    Required architecture:
      * Input conv  : (channels -> features), 3x3, padding=1, then ReLU
      * 4 middle blocks, each: Conv(features->features, 3x3, padding=1)
                               -> BatchNorm2d(features) -> ReLU
      * Output conv : (features -> channels), 3x3, padding=1
      * forward(x)  : return x - net(x)        # residual / noise-prediction
    """
    def __init__(self, channels=1, features=48):
        super().__init__()
        # TODO: build the layer list described above and wrap it in nn.Sequential.
        # TODO: assign your sequential model to `self.net`.
        layers = [nn.Conv2d(channels, features, 3, padding=1), nn.ReLU()]
        for _ in range(4):
            layers += [nn.Conv2d(features, features, 3, padding=1),nn.BatchNorm2d(features), nn.ReLU()]

        layers.append(nn.Conv2d(features, channels, 3, padding=1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # TODO: implement residual learning -- return x - self.net(x).
        return x - self.net(x)

# Instantiate the model
model = TinyDnCNN()

# --- 5. Apply the (untrained) CNN to `noisy_img` --------------------------
# TODO:
#   (a) Convert `noisy_img` to a float32 torch tensor of shape (1, 1, H, W).
#   (b) Inside a `torch.no_grad()` block, forward it through `model`.
#   (c) Convert the output back to a 2D numpy array `denoised_cnn`.

tensor_in = torch.tensor(noisy_img[None, None], dtype=torch.float32)
with torch.no_grad():
    denoised_cnn = model(tensor_in).squeeze().numpy()

# --- 6. Plot results -------------------------------------------------------
# TODO: Plot a 1x5 row in this order:
#         original `img`, `noisy_img`, `denoised_nlm`, `denoised_bm3d` (or a
#         "BM3D not available" placeholder if `bm3d_available` is False), and
#         `denoised_cnn`. Add a figure title saying "Black-box denoisers".
titles = ['Original', 'Noisy', 'NLM', 'BM3D' if bm3d_available else 'BM3D (N/A)', 'CNN (untrained)']
images = [img, noisy_img, denoised_nlm, denoised_bm3d if bm3d_available else np.full_like(img, 0.5), denoised_cnn]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Black-box denoisers')
for ax, im, title in zip(axes, images, titles):
    ax.imshow(im, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title); ax.axis('off')
plt.tight_layout()
plt.show()

### Section 5 — Questions

1. Explain in 2–3 sentences how **NLM** chooses how strongly to mix two pixels. What property of natural images does NLM exploit?
2. **BM3D** stacks similar patches into a 3D group and filters in a transform domain. Why is "filtering 50 stacked patches together" more powerful than "filtering each patch alone"? Give an intuitive answer in terms of signal vs. noise.
3. Your `TinyDnCNN` uses **residual learning**: the network predicts noise, not the clean image. The model's `forward` returns `x - self.net(x)`. Why is predicting the noise often easier than predicting the clean image directly? (Think about what the *expected output* of the last layer looks like in both cases.)
4. Run the cell. The untrained CNN output looks like random scribbles. **Explain why** in your own words. What would change after training on (noisy, clean) pairs?

*Your answers:*

**1.** NLM computes a weighted average of all pixels $j$ when reconstructing pixel $i$:
$$
\hat{x}_i = \frac{\sum_j w_{ij}\, y_j}{\sum_j w_{ij}}, \qquad
w_{ij} = \exp\!\left(-\frac{\|P_i - P_j\|_2^2}{h^2}\right),
$$
where $P_i, P_j$ are small patches centred at $i$ and $j$, and $h$ controls the decay.
Pixels whose surrounding patches look similar receive high weight; dissimilar patches are
down-weighted exponentially. NLM exploits the self-similarity of natural images:
the same texture or edge pattern tends to recur at multiple locations.

---
**2.** A single noisy patch has signal-to-noise ratio $\mathrm{SNR} \propto 1$.
Stacking $K$ similar patches and filtering the resulting 3-D array boosts the SNR:
the true signal is nearly identical across all $K$ slices, so it reinforces coherently,
while the i.i.d.\ noise averages out.
After averaging $K$ aligned copies of a signal $s$ each corrupted by
$\mathcal{N}(0,\sigma^2)$ noise, the noise standard deviation drops to
$\sigma/\sqrt{K}$, giving
$$
\mathrm{SNR}_{\text{group}} \approx \sqrt{K}\cdot \mathrm{SNR}_{\text{single}}.
$$
Filtering in a sparse transform domain (3-D hard thresholding) then kills the
residual noise while preserving the shared structure.

---
**3.** In direct learning the network must map $y \mapsto x$, so the target of the last layer
is a full natural image — a complex, high-dynamic-range function.
In residual learning the target is $n = y - x$, which has
$
\mathbb{E}[n] = 0, \qquad \mathrm{Var}(n) = \sigma^2 \ll \mathrm{Var}(x).
$
The noise is close to zero almost everywhere, so the network only needs to learn small,
localised corrections rather than reconstruct every pixel from scratch.
This makes the loss smoother and the optimisation problem easier to solve.

---
**4.** Without training, the convolutional weights are random (typically drawn from
$\mathcal{N}(0, \sigma_w^2)$), so $\mathrm{net}(x)$ is an arbitrary nonlinear function
of the input with no meaningful structure.
The output $x - \mathrm{net}(x)$ therefore contains both the original noise
and the random filter response superimposed — structured artefacts that reflect
the receptive-field geometry of the random kernels.
After training on (noisy, clean) pairs, the weights are optimised so that
$\mathrm{net}(x) \approx n$, and the forward pass $x - \mathrm{net}(x) \approx x_{\text{clean}}$
produces a smooth, noise-free image.


## 6. Plug-and-Play: the full ADMM loop

Combine everything. The full PnP-ADMM iteration is just three lines:

1. **$x$-update (data step):** $\;\; x^{k+1} \leftarrow$ `x_update_fft(y, H, rho, v - u)`
2. **$v$-update (denoise step):** $\;\; v^{k+1} \leftarrow D_\sigma(x^{k+1} + u^k)$
3. **$u$-update (dual update):** $\;\; u^{k+1} \leftarrow u^k + x^{k+1} - v^{k+1}$

You repeat until $x$ and $v$ stop changing.

### Diagnostics to track

* **PSNR** of $x$ vs. the ground truth `img` (we have it because this is a synthetic experiment): higher is better.
* **Primal residual** $r^k = \|x^{k+1} - v^{k+1}\|_2$ — measures disagreement between $x$ and $v$. Should $\to 0$.
* **Dual residual** $s^k = \rho \|v^{k+1} - v^k\|_2$ — measures stability of $v$. Should $\to 0$.

### What you will do

Implement the full loop with NLM as the plug-in denoiser, track those three numbers each iteration, and plot:

* the reconstruction next to `img` and `y`,
* PSNR vs. iteration,
* primal & dual residuals vs. iteration.


In [ ]:
# =============================================================
# Section 6 — Full PnP-ADMM loop
# =============================================================
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.restoration import denoise_nl_means, estimate_sigma
import matplotlib.pyplot as plt
import cv2

img = img.astype(np.float64)
if img.max() > 1.0:
    img = img / 255.0

# --- 1. Hyperparameters ---------------------------------------------------
# TODO: Pick a penalty parameter `rho` (start with 0.02) and a number of
#       iterations `max_iters` (try 30).
rho = 0.02
max_iters = 30

# --- 2. Initialize ADMM variables ----------------------------------------
# TODO: Initialize x, v, u.
#   x  : start from the blurry observation y.copy()
#   v  : start equal to x
#   u  : zeros, same shape as x
x = y.copy()
v = x.copy()
u = np.zeros_like(x)

# History containers for plotting
history = {'psnr': [], 'r_norm': [], 's_norm': []}

print(f"Starting PnP-ADMM for {max_iters} iterations...")

# --- 3. Main loop ---------------------------------------------------------
for k in range(max_iters):
    # Keep a copy of v from the previous iter for the dual residual
    v_prev = v.copy()

    # === Step 1: x-update (data step) ===
    # TODO: Build z = v - u, then call x_update_fft(y, H, rho, z) and assign
    #       the result back to x.
    z = v - u
    x = x_update_fft(y, H, rho, z)

    # === Step 2: v-update (PnP denoiser step) ===
    # TODO: Apply NLM to (x + u). Use float64. Pick patch_size/patch_distance/h.
    #       Assign the result to v.

    v = denoise_nl_means((x + u).astype(np.float64), patch_size=5, patch_distance=6, h=0.08, fast_mode=True)

    # === Step 3: u-update (dual update) ===
    # TODO: u = u + x - v
    u = u + x - v

    # === Diagnostics ===
    # TODO: Compute the primal residual r_norm = ||x - v||_2.
    # TODO: Compute the dual  residual s_norm = rho * ||v - v_prev||_2.
    # TODO: Compute current PSNR using psnr(img, np.clip(x, 0, 1)).
    # TODO: Append all three to `history`.

    r_norm = np.linalg.norm(x - v)
    s_norm = rho * np.linalg.norm(v - v_prev)
    current_psnr = psnr(img, np.clip(x, 0, 1))

    history['r_norm'].append(r_norm)
    history['s_norm'].append(s_norm)
    history['psnr'].append(current_psnr)

    if k % 5 == 0 or k == max_iters - 1:
        # TODO: print a short progress line that shows k, PSNR, r_norm, s_norm.
        print(f"Iter {k:02d}: PSNR = {current_psnr:.2f} dB, r_norm = {r_norm:.4f}, s_norm = {s_norm:.4f}")

print("...Loop finished.")

# --- 4. Final visualization ----------------------------------------------
# TODO: Show three images in a 1x3 row: original `img`, measured `y` with its
#       PSNR in the title, and the reconstruction np.clip(x, 0, 1) with its
#       PSNR in the title.
# TODO: Make a 1x2 figure showing:
#   left  : history['psnr'] over iterations
#   right : history['r_norm'] and history['s_norm'] over iterations (legend!)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('Original img'); ax[0].axis('off')
ax[1].imshow(y, cmap='gray'); ax[1].set_title(f'Measured y\nPSNR: {psnr(img, np.clip(y, 0, 1)):.2f}'); ax[1].axis('off')
ax[2].imshow(np.clip(x, 0, 1), cmap='gray'); ax[2].set_title(f'PnP-ADMM Recon\nPSNR: {current_psnr:.2f}'); ax[2].axis('off')
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['psnr'], color='green')
ax[0].set_title('PSNR vs. Iteration')
ax[0].set_xlabel('Iteration')
ax[0].set_ylabel('PSNR (dB)')

ax[1].plot(history['r_norm'], label='r_norm (Primal)', linestyle='--')
ax[1].plot(history['s_norm'], label='s_norm (Dual)', linestyle=':')
ax[1].set_title('Residuals vs. Iteration')
ax[1].set_xlabel('Iteration')
ax[1].legend()
plt.show()

### Section 6 — Questions

1. Run the loop and report the final PSNR. Compare it to the PSNR of the measured image $y$. Roughly how many dB did PnP-ADMM gain?
2. Try **three** values of `rho`: $0.005$, $0.02$, $0.1$ (re-run the cell each time). Describe what changes in (a) the final PSNR and (b) the shape of the residual curves. Which value gives the best result on this problem?
3. Try replacing the NLM `h` parameter with `h=0.01` (a *weak* denoiser) and re-run. What goes wrong? Now try `h=0.2` (an *aggressive* denoiser). What goes wrong now? Use this to explain how the strength of the denoiser balances the data term.
4. The dual residual $s^k = \rho \|v^{k+1} - v^k\|_2$ contains $\rho$ as a factor. Why is the factor of $\rho$ there? (Hint: it makes $r$ and $s$ comparable in scale.)
5. Suppose you didn't have access to the clean `img` — could you still tell the algorithm has converged? Which diagnostic(s) would you watch and what would they look like?

*Your answers:*

**1.**
The measured image $y$ sits at a degraded baseline of  **$25.19 \text{ dB}$**. After running $30$ iterations of the PnP-ADMM loop, the final reconstruction $x$ converges to **$30.74 \text{ dB}$**.
This yields a substantial performance jump of **$5.55 \text{ dB}$**. The algorithm successfully decouples the deblurring task (inverting the blur kernel $H$ via the FFT step) from the denoising task.

---

**2.**
* **$\rho = 0.005$ (Too Small):**
  * *(a) PSNR:* Climbs incredibly slowly, failing to reach its peak potential within $30$ iterations.
  * *(b) Residual Curves:* The primal residual $r^k$ stays high and decays at a slow rate because the algorithm places too little penalty on the mismatch between $x$ and $v$. The data fidelity term completely dominates.
* **$\rho = 0.02$ (Optimal Sweet Spot):**
  * *(a) PSNR:* Rapidly ascends and stably levels off at the maximum achievable decibel value.
  * *(b) Residual Curves:* Both the primal residual $r^k$ and dual residual $s^k$ decay smoothly and concurrently on a logarithmic scale, indicating balanced optimization progress.
* **$\rho = 0.1$ (Too Large):**
  * *(a) PSNR:* Rises fast in the first 2–3 iterations but plateaus prematurely at a lower, suboptimal final value, producing an overly smoothed image.
  * *(b) Residual Curves:* The primal residual $r^k$ drops to near-zero almost instantly because the loop aggressively punishes any disagreement. However, the dual residual $s^k$ exhibits oscillatory behavior.

**Conclusion:** The intermediate value of **$\rho = 0.02$** gives the best results.

---
**3.**

* **Weak Denoiser ($h = 0.01$):** The denoiser treats actual noise as valid structural texture, passing high-frequency noise right back into the loop. As a result, the optimization fails to clean the image, leaving heavy noise and ringing artifacts.

* **Aggressive Denoiser ($h = 0.2$):** The denoiser aggressively wipes out high-frequency content indiscriminately, treating actual structural details (like thin lines or textures) as noise. It passes an overly blurry image back to the data step, leading to an over-smoothed reconstruction.

* **The Optimization Balance:** The data fidelity step tries to invert the blur kernel $H$ (which inherently amplifies noise), while the denoiser steps in to regularize the image back onto a clean, natural image manifold. If $h$ is too small, regularization fails; if $h$ is too large, the data term's inversion efforts are completely erased by over-smoothing.

---
**4.**

The inclusion of $\rho$ emerges directly from the first-order stationarity conditions of the ADMM subproblems. The primal residual measures feasibility in the primal image space:

$$r^k = \|x^{k+1} - v^{k+1}\|_2$$

Because the dual variable update step size scales directly with the penalty parameter $\rho$:

$$u^{k+1} = u^k + x^{k+1} - v^{k+1}$$

the true variations occurring in the dual subgradient space scale directly with $\rho$. Multiplying $\|v^{k+1} - v^k\|_2$ by $\rho$ scales the dual residual appropriately, making $r^k$ and $s^k$ dimensionally and numerically comparable. Without this factor, changing $\rho$ would artificially warp convergence tracking magnitudes without representing true algorithmic state changes.

---
**5.** 

**Yes, we can tell.** In real-world deployments, the clean ground truth `img` is never available, meaning we cannot monitor PSNR. 

Instead, we can watch the **Primal Residual ($r^k$)** and **Dual Residual ($s^k$)**:
When plotted on a semi-logarithmic $y$-axis, both curves should trend downward asymptotically and flatten out into a stable, low-magnitude horizontal plateau close to zero.
Once both $r^k \to 0$ (meaning $x$ and $v$ have reached a consensus image) and $s^k \to 0$ (meaning the image states have stabilized and are no longer shifting between iterations), the optimization loop has reached a stationary fixed point and has successfully converged.


## 7. Training your own denoiser $\tilde H$

General-purpose denoisers (NLM, BM3D) don't know anything about your specific image domain. If you work with medical scans, microscopy, satellite imagery, … you can train your own predictor $\tilde H_\theta$ that is **specialized** for that domain.

### Supervised denoiser training recipe

1. Start from a collection of clean ground-truth images $X$.
2. Generate noisy inputs $V = X + W$ where $W \sim \mathcal N(0, \sigma^2 I)$. Use the *same* noise distribution you expect at inference time.
3. Train $\tilde H_\theta$ to minimize:

   $$\mathcal L(\theta) = \mathbb E_{X, W}\big[\big\| \tilde H_\theta(X + W) - X \big\|_2^2\big].$$

4. Optimize $\theta$ with SGD / Adam / etc.

The trained network *implicitly* learns the prior $h$ of your image domain.

### Why train on **patches**?

* You probably have very few full images. Random 64×64 patches turn one image into thousands of training samples.
* CNNs are translation-equivariant, so patch training generalizes to full images at inference.
* GPU memory is finite — patches fit; full 4K images may not.

### What you will do

* Build a `RandomPatchDataset` that crops random patches from a list of clean images.
* Write the training loop: sample a batch, add fresh noise on every batch, forward, MSE loss, backward, step.
* Save the weights with `torch.save(model.state_dict(), "tiny_dncnn_demo.pth")`.
* In a second cell, **load** the weights and compare your trained CNN to NLM on a fresh noisy test image.


In [ ]:
# =============================================================
# Section 7 — Train a tiny denoiser CNN
# =============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

# We reuse the TinyDnCNN class from Section 5. If it isn't visible in this
# kernel, paste it here too.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Training on:", device)

# --- 1. Random patch dataset ---------------------------------------------
class RandomPatchDataset(Dataset):
    """Yields random patches from a list of clean images.

    __getitem__ should:
      * pick a random image from `self.images`,
      * crop a random `patch_size x patch_size` patch,
      * add Gaussian noise of std `self.noise_std` to it,
      * return (noisy_patch_tensor, clean_patch_tensor), both with shape
        (1, patch_size, patch_size) and dtype float32.
    """
    def __init__(self, images, patch_size=64, noise_std=0.05, length=2000):
        # `images` is a list of 2D float numpy arrays in [0, 1].
        self.images = images
        self.patch_size = patch_size
        self.noise_std = noise_std
        self.length = length

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # TODO: implement the four steps in the docstring above.
        # Hints:
        #   - use np.random.randint to pick an image index and crop offsets
        #   - convert to torch with torch.from_numpy(...).float().unsqueeze(0)
        img = self.images[np.random.randint(len(self.images))]
        h, w = img.shape
        r = np.random.randint(0, h - self.patch_size)
        c = np.random.randint(0, w - self.patch_size)
        clean = img[r:r+self.patch_size, c:c+self.patch_size]
        noisy = clean + np.random.normal(0, self.noise_std, clean.shape)
        to_t = lambda x: torch.from_numpy(x.astype(np.float32)).unsqueeze(0)
        return to_t(noisy), to_t(clean)

# --- 2. Build dataset + dataloader ---------------------------------------
# We only have one image (`img`) handy. That's enough for a demo.
# TODO: Build a RandomPatchDataset with images=[img], a sensible patch_size,
#       noise_std (e.g. 0.05), and length (e.g. 2000). Wrap it in a DataLoader
#       with batch_size around 16 and shuffle=True.
dataset = RandomPatchDataset(images=[img], patch_size=64, noise_std=0.05, length=2000)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# --- 3. Training loop ----------------------------------------------------
# TODO: Instantiate a fresh TinyDnCNN on `device`, an Adam optimizer with
#       lr=1e-3, and an nn.MSELoss criterion.
model = TinyDnCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

num_epochs = 5  # keep it small for a demo
for epoch in range(num_epochs):
    epoch_loss = 0.0
    n_batches = 0
    for noisy_patches, clean_patches in dataloader:
        noisy_patches = noisy_patches.to(device)
        clean_patches = clean_patches.to(device)
        # TODO: forward pass to get the predicted clean patches.
        # TODO: compute MSE loss between prediction and clean_patches.
        # TODO: zero the optimizer grads, call loss.backward(), optimizer.step().
        # TODO: accumulate epoch_loss and increment n_batches.
        pred = model(noisy_patches)
        loss = criterion(pred, clean_patches)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    avg = epoch_loss / max(n_batches, 1)
    print(f"epoch {epoch+1}/{num_epochs}, avg MSE: {avg:.5f}")

# --- 4. Save the trained weights ----------------------------------------
# TODO: torch.save(model.state_dict(), 'tiny_dncnn_demo.pth')
torch.save(model.state_dict(), 'tiny_dncnn_demo.pth')

In [ ]:
# =============================================================
# Section 7 (cont.) — Use the trained denoiser, compare with NLM
# =============================================================
import torch
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.restoration import denoise_nl_means

# --- 1. Reinstantiate the model and load weights -------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# TODO: instantiate `trained_model = TinyDnCNN().to(device)` and call
#       trained_model.load_state_dict(torch.load('tiny_dncnn_demo.pth')).
trained_model = TinyDnCNN().to(device)
trained_model.load_state_dict(torch.load('tiny_dncnn_demo.pth', map_location=device))

# TODO: Set the model to evaluation mode (.eval()) so BatchNorm uses running stats.
trained_model.eval()

# --- 2. Build a fresh test image -----------------------------------------
# TODO: Pick a test_noise_std (try 0.05), seed numpy with np.random.seed(42)
#       for reproducibility, and create `noisy_test_img` = img + noise,
#       clipped to [0, 1].
test_noise_std = 0.05
np.random.seed(42)
noisy_test_img = np.clip(img + test_noise_std * np.random.randn(*img.shape), 0, 1)

# --- 3. Denoise with the trained CNN -------------------------------------
# TODO:
#   (a) Convert `noisy_test_img` to a (1, 1, H, W) float32 tensor on `device`.
#   (b) Inside torch.no_grad(), pass it through `trained_model`.
#   (c) Bring the result back to a 2D numpy array `denoised_cnn`.
tensor_noisy_test = torch.from_numpy(noisy_test_img).float().unsqueeze(0).unsqueeze(0).to(device)
with torch.no_grad():
    tensor_cnn_out = trained_model(tensor_noisy_test)
denoised_cnn = np.clip(tensor_cnn_out.squeeze().cpu().numpy(), 0, 1)

# --- 4. Denoise with NLM for comparison ----------------------------------
# TODO: Run denoise_nl_means on noisy_test_img with reasonable hyperparams.
denoised_nlm = denoise_nl_means(noisy_test_img.astype(np.float64), patch_size=5, patch_distance=6, h=0.08, fast_mode=True)
denoised_nlm = np.clip(denoised_nlm, 0, 1)

# --- 5. Compare with PSNR + plot -----------------------------------------
# TODO: Compute psnr_noisy, psnr_nlm, psnr_cnn (use np.clip(..., 0, 1) on
#       the denoiser outputs to keep values in range).
# TODO: Print all three numbers with labels.
# TODO: Plot a 1x4 row: original, noisy, NLM result, CNN result. Show each
#       PSNR in the title.
psnr_noisy = psnr(img, noisy_test_img, data_range=1.0)
psnr_nlm   = psnr(img, denoised_nlm,   data_range=1.0)
psnr_cnn   = psnr(img, denoised_cnn,   data_range=1.0)

print(f"Noisy : {psnr_noisy:.2f} dB")
print(f"NLM   : {psnr_nlm:.2f} dB")
print(f"CNN   : {psnr_cnn:.2f} dB")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, image, title in zip(axes,[img, noisy_test_img, denoised_nlm, denoised_cnn],
    [f"Original", f"Noisy\n{psnr_noisy:.2f} dB",
     f"NLM\n{psnr_nlm:.2f} dB", f"CNN\n{psnr_cnn:.2f} dB"]):
    ax.imshow(image, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title); ax.axis('off')
plt.tight_layout()
plt.show()

### Section 7 — Questions

1. We generate the noise **on every batch** (inside `__getitem__`), not once at dataset construction. Why? What kind of overfitting does this prevent, and what does the network learn instead?
2. We train on random **patches** instead of full images. List two practical reasons (one about data quantity, one about compute) why this is the standard recipe.
3. Switch the model to **eval()** mode before inference. What specifically changes about (a) BatchNorm layers and (b) Dropout layers? Why would forgetting `eval()` give wrong PSNR numbers?
4. Compare your trained `TinyDnCNN` PSNR to NLM's PSNR. Which is better on your run? Now think about a fair-comparison caveat: we only trained on ONE image. How would you expect the comparison to change if you trained on 1000 images from a target domain?
5. Inside the PnP-ADMM loop the denoiser sees inputs of the form $x^{k+1} + u^k$, NOT clean-image + Gaussian noise. Is training with simple Gaussian noise still a reasonable surrogate? Argue both sides briefly.

*Your answers:*

**1.** Each epoch the same clean patch receives a freshly sampled noise realization.
The network cannot memorize specific noisy images so it must learn the
underlying clean-image prior $p(x)$ because the noise instance
$n \sim \mathcal{N}(0, \sigma^2 I)$ changes every pass.

---

**2.**  
Data quantity: A single $H{\times}W$ image yields $\sim\!\frac{HW}{p^2}$ non-overlapping $p{\times}p$ patches,multiplying the training set size without new data collection.

Compute: Small patches allow a batch size $B\!>\!1$,stabilising gradient estimates; full images would saturate GPUmemory and force $B{=}1$.

---       
**3.** $\textbf{(a) BatchNorm:}$ In train() mode the layer normalises
with batch statistics $\hat\mu_B, \hat\sigma_B^2$.
In eval() mode it uses the exponential-moving-average
(running) statistics $\mu_r, \sigma_r^2$ accumulated over training,
which are stable estimates of the population moments.

$\textbf{(b) Dropout:}$ In train() mode each neuron is zeroed
with probability $p$ independently per forward pass.
In eval() mode the mask is the identity; all activations
pass through, giving a deterministic output.

Forgetting eval() corrupts the normalisation in BatchNorm
and injects stochastic dropout noise, both degrading and
randomising the denoised output, hence lowering PSNR.

---
**4.** The trained TinyDnCNN edges out NLM slightly on this specific test. Training on a single image means the learned prior
$p_\theta(x)$ is biased toward one texture.
With $N=1000$ domain images the network approximates the true
domain prior and generalises.

---

**5.** For: At early iterations $u^k \approx \mathbf{0}$, so
$\tilde{v} = x^{k+1} + u^k \approx x_{\text{true}} + \epsilon$
with $\epsilon$ noise-like; Gaussian training is a valid surrogate.

Against: The dual variable satisfies
$u^{k+1} = u^k + x^{k+1} - v^{k+1}$,
accumulating systematic, spatially-correlated bias.
The effective perturbation is not i.i.d.\ Gaussian, so a
purely Gaussian-trained denoiser can under-correct structured
residuals introduced by $u^k$.


## 8. The zero-curl condition

### Intuition

* Think of $h(x)$ as a "landscape" / potential and the gradient $\nabla h$ as the slope.
* The denoiser's correction $F(v) = v - D(v)$ is the **vector field** that pushes the noisy point $v$ toward its denoised version $D(v)$.
* For $D$ to be a *true* proximal map of some scalar function $\phi$, we need $F(v) = \nabla \phi(v)$ — i.e. $F$ must be a **conservative** vector field.
* A necessary condition for a vector field to be conservative is **zero curl**: $\nabla \times F = 0$.

If $D$ is a learned CNN, $F$ generally does **not** have zero curl. That is one reason rigorous PnP convergence proofs need extra assumptions.

### A warning about the code

The code in the lecture takes a 2D *image* $F = v - D(v)$, computes its 2D gradient, and then computes the curl. **That is not the high-dimensional vector field of the operator $D$.** It is testing a different identity:

$$\nabla \times \nabla F = 0 \quad \text{(curl of a gradient is identically zero)}.$$

So the code below is really a **numerical precision test** of `np.gradient`. In the questions you will be asked how one *could* actually test the zero-curl property of $D$.

### What you will do

Compute the difference image $F = v - D(v)$, its gradient, then `curl(grad(F))`, and verify it is numerically (almost) zero.


In [ ]:
# =============================================================
# Section 8 — A numerical curl(grad(F)) test
# =============================================================
import numpy as np
import matplotlib.pyplot as plt
from skimage.restoration import denoise_nl_means

# --- 1. Build the difference image F = v - D(v) --------------------------
# TODO: Create a noisy image v = img + Gaussian noise of std 0.03.
# TODO: Apply denoise_nl_means to v to get Dv. Use float64.
# TODO: Define F = v - Dv.

v = img + np.random.normal(0, 0.03, img.shape)
v = np.clip(v, 0, 1)

v = v.astype(np.float64)

sigma_est = estimate_sigma(v, channel_axis=None)
Dv = denoise_nl_means(
    v, 
    patch_size=5, 
    patch_distance=3, 
    h=1.15 * sigma_est, 
    fast_mode=True,
    preserve_range=True
)

F = v - Dv

# --- 2. Take the 2D gradient of F ---------------------------------------
# np.gradient returns (dF/d_axis0, dF/d_axis1).
# TODO: Fx, Fy = np.gradient(F)
Fx, Fy = np.gradient(F)

# --- 3. Compute the curl_z of the gradient field (Fx, Fy) ----------------
# curl_z = d(Fy)/d_axis0 - d(Fx)/d_axis1
# TODO: implement using np.gradient(..., axis=...).
dFy_dx = np.gradient(Fy, axis=0)
dFx_dy = np.gradient(Fx, axis=1)
curl = dFy_dx - dFx_dy

# --- 4. Report ------------------------------------------------------------
# TODO: Print the mean absolute curl. It must be ~ 1e-16 (floating-point error)
#       because curl of a gradient is mathematically zero.
print(f"Mean absolute curl : {np.mean(np.abs(curl)):.3e}")
print(f"Max  absolute curl : {np.max(np.abs(curl)):.3e}")

# --- 5. Visualize ---------------------------------------------------------
# TODO: Make a 1x2 figure showing F (use cmap='seismic', colorbar) and `curl`
#       (also cmap='seismic', colorbar). Title each subplot.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(F, cmap='seismic',
                     vmin=-np.abs(F).max(), vmax=np.abs(F).max())
axes[0].set_title(r'$F = v - D(v)$  (residual image)', fontsize=13)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(curl, cmap='seismic', vmin=-np.abs(curl).max() - 1e-18, vmax=np.abs(curl).max() + 1e-18)
axes[1].set_title(r'$\nabla \times \nabla F$  (curl of gradient)', fontsize=13)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.suptitle(r'curl$(\nabla F) \approx 0$  — numerical precision test', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Section 8 — Questions

1. Prove (in 3–4 lines of math) that **the curl of a gradient is always zero** for a smooth scalar field $F$. State which equality of mixed partial derivatives you are using.
2. The code computes `curl(grad(F))`. Explain in your own words why this is *not* a test of the denoiser's zero-curl property, even though the lecture text suggests it.
3. **Design (in words, no code)** an experiment that *actually* tests whether the operator $F(v) = v - D(v)$ is curl-free.
   * Hint: pick two pixels $i, j$, build a small loop in pixel space, and check whether the line integral of $F$ around the loop is zero.
   * Bonus: write down the two-pixel finite-difference test as a single equation.
4. Suppose your experiment from question 3 detects non-zero curl for some learned denoiser $D$. What does this **fail to imply** about PnP convergence? (Be careful: zero-curl is *necessary* for $D$ to be a proximal map, but is it *sufficient* for PnP convergence?)

*Your answers:*

**1.**
For a smooth 2D scalar field $F(x, y)$, the gradient is $\nabla F = \begin{bmatrix} \frac{\partial F}{\partial x} \\ \frac{\partial F}{\partial y} \end{bmatrix}$. Its curl is:
$$ \nabla \times \nabla F = \frac{\partial}{\partial x}\left(\frac{\partial F}{\partial y}\right) - \frac{\partial}{\partial y}\left(\frac{\partial F}{\partial x}\right) = \frac{\partial^2 F}{\partial x \partial y} - \frac{\partial^2 F}{\partial y \partial x} $$
By Clairaut's Theorem on the equality of mixed partial derivatives ($\frac{\partial^2 F}{\partial x \partial y} = \frac{\partial^2 F}{\partial y \partial x}$), this expression is identically zero:
$$ \nabla \times \nabla F = 0 $$

---

**2.**
The denoiser operator $D(v)$ maps an entire high-dimensional image vector $v \in \mathbb{R}^N$ to another image vector. The true conservative vector field property concerns the derivatives across the image space $\mathbb{R}^N$ (i.e., verifying if $\frac{\partial F_i}{\partial v_j} = \frac{\partial F_j}{\partial v_i}$ for two different pixel variables). 

Instead, the lecture code treats a single static 2D residual image $F$ as a spatial coordinate system, computing spatial gradients with respect to pixel coordinates $(x, y)$. Because it evaluates $\nabla \times \nabla F$, it is simply verifying an analytical calculus identity on a 2D grid surface (a numerical precision test of $\texttt{np.gradient}$). It reveals absolutely nothing about how $D(v)$ behaves when the input image vector $v$ itself is perturbed.

---
**3.** 
Experimental Design: Pick a base image $v \in \mathbb{R}^N$. Select two distinct pixels indexed at $i$ and $j$. We approximate the cross-pixel sensitivities using finite differences:

   .Add a tiny perturbation $\epsilon$ to pixel $j$ to create $v + \epsilon e_j$, pass it through the denoiser, and measure the change at pixel $i$. This approximates $\frac{\partial F_i}{\partial v_j} \approx \frac{F_i(v + \epsilon e_j) - F_i(v)}{\epsilon}$.
   .Conversely, perturb pixel $i$ by $\epsilon$ to create $v + \epsilon e_i$, and measure the change at pixel $j$ to approximate $\frac{\partial F_j}{\partial v_i} \approx \frac{F_j(v + \epsilon e_i) - F_j(v)}{\epsilon}$.
   .Compare the two values to see if they match within a tolerance threshold.

   $\textbf{Bonus (Single Equation Test):}$ By clearing the $\epsilon$ denominator, the finite-difference check for a symmetric Jacobian along a closed two-pixel loop is:
   $$ F_i(v + \epsilon e_j) - F_i(v) \approx F_j(v + \epsilon e_i) - F_j(v) $$

---
**4.**
If the experiment detects a non-zero curl, it formally proves that $D(v)$ is not a proximal operator of any underlying scalar objective function $\phi$. 

However, this fails to imply that PnP will diverge. While being a proximal map (zero-curl) makes convergence straightforward to prove via classical convex optimization frameworks, it is a necessary condition for proximaity, but not a necessary condition for PnP convergence. Modern PnP algorithms can be proven to converge using entirely separate mathematical properties. For instance, if the denoiser $D$ can be shown to be a contraction mapping, or if it satisfies Averaged/Monotone Operator splitting conditions (such as bounded denoiser residuals via a Schur-complement technique), the PnP algorithm will successfully converge regardless of a non-zero curl.


## 9. Stacked operators — an assembly line of priors

Sometimes a single denoiser is not enough. A **stacked operator** chains several specialized modules:

$$\mathcal T(x) = (D_2 \circ M \circ D_1)(x) = D_2\!\left(M\!\left(D_1(x)\right)\right).$$

Reading right to left: $D_1$ removes the worst noise, $M$ enhances contrast, $D_2$ smooths the small artifacts $M$ created.

### Why stack?

* Different modules are good at different jobs (denoising, deblurring, contrast, color, …).
* You can prototype quickly: each block is a black box.

### Costs

* The composition is **order-sensitive**: $D_2 \circ M \circ D_1 \neq D_1 \circ M \circ D_2$.
* Convergence proofs become much harder; fixed-point and equilibrium notions (Sections 10–11) are how the literature reasons about stacks.

### What you will do

Build $D_1$ = NLM, $M$ = a contrast stretch you write yourself, $D_2$ = TV denoising, and visualize the output of each stage.


In [ ]:
# =============================================================
# Section 9 — A 3-step stacked operator
# =============================================================
import numpy as np
import matplotlib.pyplot as plt
from skimage.restoration import denoise_nl_means, denoise_tv_chambolle

# --- 1. Implement the contrast-stretch module M --------------------------
def contrast_stretch(x):
    """Linearly stretch x so that its 2nd percentile maps to 0 and its 98th
    percentile maps to 1; clip everything to [0, 1].

    TODO: implement using np.percentile and np.clip. Return `x` unchanged
          if the dynamic range (b - a) is zero.
    """
    # TODO
    a = np.percentile(x, 2)   
    b = np.percentile(x, 98) 
    if (b - a) == 0:
        return x
    stretched = (x - a) / (b - a)
    return np.clip(stretched, 0.0, 1.0)

# --- 2. Define the three operators ---------------------------------------
# TODO: D1 = a lambda that runs denoise_nl_means with sensible hyperparams.
sigma_est = estimate_sigma(img, channel_axis=None) if 'img' in locals() else 0.04
D1 = lambda x: denoise_nl_means(
    x, 
    patch_size=5, 
    patch_distance=3, 
    h=1.15 * (estimate_sigma(x, channel_axis=None) if x.ndim == 2 else 0.04), 
    fast_mode=True, 
    preserve_range=True
)
# TODO: M = a lambda (or function) that calls your contrast_stretch.
M = lambda x: contrast_stretch(x)

# TODO: D2 = a lambda that calls denoise_tv_chambolle with a small `weight`
#       (try 0.08). TV smooths blocky artifacts while keeping edges.
D2 = lambda x: denoise_tv_chambolle(x, weight=0.001)

# --- 3. Build a noisy input ----------------------------------------------
# TODO: noisy_img = img + Gaussian noise of std 0.04, clipped to [0, 1].

noisy_img = img + np.random.normal(0, 0.04, img.shape)
noisy_img = np.clip(noisy_img, 0, 1)

# --- 4. Run the stack step by step ---------------------------------------
# TODO: out_D1     = D1(noisy_img)
# TODO: out_M      = M(out_D1)
# TODO: out_stacked = D2(out_M)         # this is D2(M(D1(noisy_img)))
out_D1      = D1(noisy_img)
out_M       = M(out_D1)
out_stacked = D2(out_M)

# --- 5. Plot intermediate results ----------------------------------------
# TODO: Plot a 1x4 row: original `img`, `noisy_img`, `out_D1`, and the final
#       `out_stacked`. Add titles.

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title("Original Ground Truth", fontsize=12)
axes[0].axis('off')

axes[1].imshow(noisy_img, cmap='gray', vmin=0, vmax=1)
axes[1].set_title("Noisy Input ($\sigma = 0.04$)", fontsize=12)
axes[1].axis('off')

axes[2].imshow(out_D1, cmap='gray', vmin=0, vmax=1)
axes[2].set_title("Stage 1: NLM ($D_1$)", fontsize=12)
axes[2].axis('off')

axes[3].imshow(out_stacked, cmap='gray', vmin=0, vmax=1)
axes[3].set_title("Stage 3: Full Stack ($D_2 \circ M \circ D_1$)", fontsize=12)
axes[3].axis('off')

plt.suptitle("Pipeline Output of Stacked Operator Framework", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

### Section 9 — Questions

1. Function composition is **not** commutative. Run the stack in the reverse order $D_1 \circ M \circ D_2$ (i.e. apply $D_2$ first, then $M$, then $D_1$). Describe in 2–3 sentences how the result differs from the original order. Which order looks more visually pleasant on your image?
2. Why is it generally a bad idea to apply contrast stretch $M$ **first**, before any denoiser? Argue in terms of how $M$ would amplify the noise.
3. Suppose all three operators in the stack are non-expansive (they don't blow up distances). Is their composition guaranteed to be non-expansive? Justify briefly.
4. Give one **realistic application** (medical imaging, scientific imaging, astrophotography, …) where a stacked operator pipeline like this might be the right design choice. Identify what each of the three stages would represent in your scenario.

*Your answers:*

**1.** When running the stack in the reverse order, applying Total Variation minimization ($D_2$) first flattens out both high-frequency noise and fine textures simultaneously into blocky regions. The subsequent contrast stretch ($M$) then harshly amplifies these artificial, piecewise-constant block boundaries rather than real image details. Finally, Non-Local Means ($D_1$) treats these sharp, artificial blocks structural content, preserving them and leaving a distinct, unnaturally blocky. 

The original order ($D_2 \circ M \circ D_1$) looks significantly more visually pleasant, because $D_1$ first suppresses the pixel-level grain, allowing $M$ to safely boost the dynamic range without turning noise into hard structures, and $D_2$ cleanly smooths out any minor artifacts left at the very end.

---
**2.** Applying the contrast stretch $M$ before any denoising is a poor design choice because it dramatically amplifies the noise variance before the filters can isolate it. The linear stretching maps the 2nd percentile to 0 and the 98th percentile to 1 via division by a compressed dynamic range width $(b - a)$. If heavy noise is present, noise spikes expand this denominator or cause high-frequency fluctuations to stretch to the absolute boundaries of the dynamic range ($0$ and $1$). This scales up the high-frequency spatial gradients, making it far more difficult for subsequent edge-preserving denoisers like NLM or TV to distinguish between true image edges and heavily amplified noise structures.

---
**3.** Yes, the composition is guaranteed to be non-expansive. By definition, an operator $T$ is non-expansive if for any two vectors $u$ and $v$, it satisfies $\|T(u) - T(v)\| \leq \|u - v\|$. Given that $D_1$, $M$, and $D_2$ each independently possess a Lipschitz constant $L \leq 1$, we chain their bounding inequalities sequentially from right to left:
$$\|D_1(x) - D_1(y)\| \leq \|x - y\|$$
$$\|M(D_1(x)) - M(D_1(y))\| \leq \|D_1(x) - D_1(y)\| \leq \|x - y\|$$
$$\|D_2(M(D_1(x))) - D_2(M(D_1(y)))\| \leq \|M(D_1(x)) - M(D_1(y))\| \leq \|x - y\|$$
Thus, the total stacked operator $\mathcal{T} = D_2 \circ M \circ D_1$ satisfies $\|\mathcal{T}(x) - \mathcal{T}(y)\| \leq \|x - y\|$, proving that the composition preserves non-expansiveness.

---
**4.** A stacked operator assembly line is highly effective for processing deep-space astrophotography images (e.g., photographing a distant nebula captured by a ground-based telescope).

Stage 1 ($D_1$ - Non-Local Means): Used first to suppress the heavy thermal sensor noise and atmospheric Poisson noise present in long-exposure frames. Because NLM searches globally for structural patterns, it preserves tiny star clusters and fine galactic structurs while washing away uncorrelated random background grain.

Stage 2 ($M$ - Contrast Stretch): Deep-sky objects are incredibly faint and compressed into a narrow, dark band of the raw histogram. A percentile-based stretch is imporatant at this stage to pull the faint nebula details out of the dark background noise floor, making the structures visible to the human eye.

Stage 3 ($D_2$ - Total Variation Denoising): Boosting the contrast in Stage 2 inevitably exaggerates residual low-frequency color or background gradients. A mild TV filter is applied last to flatten out the empty space background into a smooth, clean uniform black while keeping the hard, bright boundaries of the stars and nebular gas walls sharp.



## 10. PnP and MACE — consensus equilibrium

Reframe the whole problem as a **negotiation** between two agents:

* **Agent 1 ($F_1$): the data-fidelity agent** — pulls any guess toward something that matches $y$.
* **Agent 2 ($F_2$): the prior agent** — pulls any guess toward something that looks natural (a denoiser).

They disagree. The **Consensus Equilibrium (CE)** is the single image $x^\star$ where the two pulls cancel:

$$F_1(x_1) = x_1 - u,\qquad F_2(x_2) = x_2 + u,\qquad x_1 = x_2 = x^\star.$$

**MACE = Multi-Agent Consensus Equilibrium** — a family of algorithms that find $x^\star$.

* **PnP-ADMM (Section 6)** is one of them.
* The **averaging loop** below is another, simpler, less stable one.

### A simple consensus-by-averaging algorithm

Starting from a guess $z$, at each iteration:

1. Ask Agent 1: $x_1 \leftarrow$ data step on $z$.
2. Ask Agent 2: $x_2 \leftarrow$ denoiser on $z$.
3. New $z \leftarrow \tfrac{1}{2}(x_1 + x_2)$.

It pushes $x_1$ and $x_2$ toward agreement step by step. It is **not** ADMM.

### What you will do

Implement this averaging consensus loop and watch the PSNR rise. It will probably converge **slower** than the ADMM loop in Section 6.


In [ ]:
# =============================================================
# Section 10 — Simple consensus-by-averaging loop
# =============================================================
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.restoration import denoise_nl_means

# --- 1. Hyperparameters ---------------------------------------------------
# TODO: Initial guess `z` = y.copy(). Use the SAME rho you used in Section 6
#       so the data agent behaves the same way. Pick max_iters around 40.
z = y.copy()
rho = 0.02
max_iters = 40

psnr_history_avg = []

# --- 2. Consensus by averaging -------------------------------------------
for k in range(max_iters):
    # Ask Agent 1 (data fidelity) for its best image given current `z`.
    # TODO: x1 = x_update_fft(y, H, rho, z)
    x1 = x_update_fft(y, H, rho, z) 

    # Ask Agent 2 (denoiser) for its best image given current `z`.
    # TODO: x2 = denoise_nl_means(z.astype(np.float64), ...)
    x2 = denoise_nl_means(z.astype(np.float64), patch_size=5, patch_distance=6, h=0.08, fast_mode=True)

    # New consensus = simple average of the two opinions.
    # TODO: z = 0.5 * (x1 + x2)
    z = 0.5 * (x1 + x2)  # TODO

    # Track PSNR vs. the ground truth.
    # TODO: append psnr(img, np.clip(z, 0, 1)) to psnr_history.
    current_psnr = psnr(img, np.clip(z, 0, 1))
    psnr_history_avg.append(current_psnr)

    if k % 10 == 0 or k == max_iters - 1:
        # TODO: print the iteration index and the current PSNR.
        print(f"Iter {k:02d} — PSNR: {current_psnr:.2f} dB")

# --- 3. Final visualization ----------------------------------------------
# TODO: Plot a 1x3 row: original `img`, measured `y` (PSNR in title), and
#       np.clip(z, 0, 1) (PSNR in title).

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('Original img'); ax[0].axis('off')
ax[1].imshow(y, cmap='gray'); ax[1].set_title(f'Measured y\nPSNR: {psnr(img, y):.2f}'); ax[1].axis('off')
ax[2].imshow(np.clip(z, 0, 1), cmap='gray'); ax[2].set_title(f'Consensus z\nPSNR: {current_psnr:.2f}'); ax[2].axis('off')
plt.show()

# TODO: Plot psnr_history versus iteration. Add axis labels and a title.

plt.figure(figsize=(8, 4))
plt.plot(psnr_history_avg, color='purple', linewidth=2)
plt.title('Averaging Consensus: PSNR vs. Iteration')
plt.xlabel('Iteration')
plt.ylabel('PSNR (dB)')
plt.grid(True)
plt.show()


### Section 10 — Questions

1. Compare the final PSNR of this averaging consensus loop to your PnP-ADMM result from Section 6 (same `rho`, same denoiser settings). Which converges faster? Which reaches a better final PSNR?
2. Replace the simple average with a **weighted** average $z \leftarrow \alpha\, x_1 + (1 - \alpha)\, x_2$ for $\alpha \in \{0.2, 0.5, 0.8\}$. Briefly describe how each weighting changes the final image and the PSNR.
3. Starting from the equilibrium equations
   $$F_1(x_1) = x_1 - u,\qquad F_2(x_2) = x_2 + u,\qquad x_1 = x_2 = x^\star,$$
   add the first two equations to **eliminate $u$**. What does the resulting equation say in words about the consensus point $x^\star$?
4. PnP-ADMM also finds a consensus point, but the two loops behave differently in practice. In your own words, what is the role of the dual variable $u$ in ADMM that is **missing** from the simple averaging loop, and why does that make ADMM more robust?

*Your answers:*

**1.**

PnP-ADMM converges significantly faster than the simple averaging consensus loop. 
PnP-ADMM reaches a higher and more stable final PSNR. The simple averaging loop lacks a mechanism to strictly enforce the consensus constraint ($x_1 = x_2$), causing it to stall early or over-smooth the image, limiting its peak performance.

---

**2.**

* **$\alpha = 0.2$ (Prior-heavy):** The update heavily favors the denoiser ($x_2$). The final image will become overly smooth, blurry, and lose fine structures/edges, resulting in a lower PSNR due to under-fitting the data.
* **$\alpha = 0.5$ (Balanced):** Equal weight is given to data fidelity and denoising, providing a balanced trade-off between noise removal and sharpness.
* **$\alpha = 0.8$ (Data-heavy):** The update heavily favors data fidelity ($x_1$). The final image will retain significant noise and artifacts from $y$. While it may look sharper, the lack of sufficient denoising leads to a lower PSNR.
---
**3.**

At the consensus equilibrium, we have $x_1 = x_2 = x^\star$. Substituting this into the equations gives:
$$F_1(x^\star) = x^\star - u$$
$$F_2(x^\star) = x^\star + u$$
Adding these two equations together eliminates $u$:
$$F_1(x^\star) + F_2(x^\star) = 2x^\star \implies x^\star = \frac{F_1(x^\star) + F_2(x^\star)}{2}$$

This equation states that the consensus equilibrium point $x^\star$ is the exact **arithmetic mean (average)** of the outputs of the data fidelity agent ($F_1$) and the prior agent ($F_2$). At this unique point, the opposing "pulls" exerted by both agents perfectly balance out.

---
**4.**

In PnP-ADMM, the dual variable $u$ acts as an **accumulator of past disagreements** (or errors) between the data step $x$ and the denoiser step $v$. 

If $x$ and $v$ disagree on certain pixels, those errors accumulate in $u$ over successive iterations. This builds up numerical "pressure" that forces both agents to reconcile their differences and strictly drive the primal residual $\|x - v\|_2$ to zero. The simple averaging loop has no memory of past states; it blindly averages the current outputs, which frequently causes the optimization to stall at a suboptimal compromise where $x_1 \neq x_2$.


## 11. Equilibrium equations and the fixed-point view

Condense the whole algorithm into a single operator $T$ that represents **one full iteration**:

$$x_{k+1} = T(x_k).$$

The solution we want is a **fixed point** of $T$:

$$x^\star = T(x^\star).$$

One natural way to build $T$:

* Let $G(x; y)$ be the data step (our `x_update_fft`).
* Let $D_\sigma$ be the denoising step.
* Then $T(x) = D_\sigma(G(x; y))$.

The simplest fixed-point algorithm is **Picard iteration**:

$$x_{k+1} = T(x_k) = D_\sigma\!\left(G(x_k; y)\right).$$

### Why we like this view

If we can show $T$ is a **contraction** (Lipschitz constant $L < 1$), then Banach's fixed-point theorem guarantees:

* a unique fixed point $x^\star$ exists;
* Picard iteration converges to it from any starting point;
* the convergence is geometric ($\|x_k - x^\star\| \le L^k \|x_0 - x^\star\|$).

In practice $T$ for PnP is often only **non-expansive**, not strictly contractive, but the fixed-point view still tells us when convergence is plausible.

### What you will do

Implement the Picard iteration $x_{k+1} = D(G(x_k))$ directly (no $u$, no $v$, no averaging) and track its PSNR.


In [ ]:
# =============================================================
# Section 11 — Direct fixed-point (Picard) iteration  x_{k+1} = D(G(x_k))
# =============================================================
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.restoration import denoise_nl_means

# --- 1. Define the two operators -----------------------------------------
def G_data_step(x, y, H, rho):
    """The data step. We use our FFT solver, treating the current x as z."""
    # TODO: return x_update_fft(y, H, rho=rho, z=x)
    return x_update_fft(y, H, rho, x)

def D_denoising_step(v):
    """The denoising step. We use NLM."""
    # TODO: return denoise_nl_means(v.astype(np.float64), ...)
    v_clipped = np.clip(v, 0, 1)
    return denoise_nl_means(v_clipped.astype(np.float64), h=0.08, fast_mode=True, channel_axis=None)

# --- 2. Run the Picard iteration -----------------------------------------
# TODO: x = y.copy(), rho = 0.02, max_iters = 30.
x = y.copy()         # TODO
rho = 0.02      # TODO
max_iters = 30 # TODO

psnr_history_picard = []

for k in range(max_iters):
    # TODO: x_data = G_data_step(x, y, H, rho)
    x_data = G_data_step(x, y, H, rho)
    # TODO: x      = D_denoising_step(x_data)
    x      = D_denoising_step(x_data)
    # TODO: append psnr(img, np.clip(x, 0, 1)) to psnr_history.
    current_psnr = psnr(img, np.clip(x, 0, 1), data_range=1.0)
    psnr_history_picard.append(current_psnr)

    if k % 5 == 0 or k == max_iters - 1:
        # TODO: print iteration index and the current PSNR.
        print(f"Iter {k:02d} — PSNR: {current_psnr:.2f} dB")

# --- 3. Final plots ------------------------------------------------------
# TODO: 1x3 figure: img, y (PSNR), final x (PSNR).
# TODO: separate plot of psnr_history vs. iteration.

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img, cmap='gray')
plt.title("Original img")
plt.axis('off')

plt.subplot(1, 3, 2)
psnr_y = psnr(img, np.clip(y, 0, 1), data_range=1.0)
plt.imshow(y, cmap='gray')
plt.title(f"Measured y\nPSNR: {psnr_y:.2f} dB")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(np.clip(x, 0, 1), cmap='gray')
plt.title(f"Picard Fixed-Point\nPSNR: {psnr_history_picard[-1]:.2f} dB")
plt.axis('off')

plt.tight_layout()
plt.show()

# psnr_history vs. iteration
plt.figure(figsize=(6, 4))
plt.plot(range(1, max_iters + 1), psnr_history_picard, 'o-', color='tab:red', linewidth=2)
plt.title("Picard Iteration Convergence")
plt.xlabel("Iteration")
plt.ylabel("PSNR (dB)")
plt.grid(True, linestyle="--")
plt.show()

### Section 11 — Questions

1. State the **Banach fixed-point theorem** in your own words (3–5 lines). Be precise about what "contraction" means.
2. Suppose $T(x) = \alpha\, x + (1 - \alpha)\, c$ for some constant image $c$ and scalar $0 < \alpha < 1$. By hand, find its fixed point and show that Picard iteration converges to it at a geometric rate. What is the rate?
3. Compare your three solvers — Section 6 (PnP-ADMM), Section 10 (averaging), Section 11 (Picard) — on the **same** problem (same $y$, same $H$, same $\rho$, same denoiser). Make a single plot overlaying their PSNR-vs-iteration curves. Which one converges fastest? Which one reaches the highest PSNR?
4. The Picard iteration $x_{k+1} = D(G(x_k))$ does not have a dual variable $u$. When PnP-ADMM converges, what is the *limiting value of $u$* telling us, and what information are we losing by dropping it?
5. **Anderson acceleration** is a technique that uses the last $m$ iterates to extrapolate a better next guess for fixed-point problems. Conceptually, why might it help here? (No code needed.)

*Your answers:*

**1.**
The theorem states that if a mapping $T: X \to X$ on a complete metric space $X$ is a **contraction**, it possesses a unique fixed point $x^\star \in X$ (where $T(x^\star) = x^\star$). So starting from any arbitrary initial guess $x_0$, the Picard iteration sequence $x_{k+1} = T(x_k)$ is guaranteed to converge to $x^\star$. 

An operator is a *contraction* if it strictly shrinks the distance between any two points it maps. Mathematically, there must exist a Lipschitz constant $L \in [0, 1)$ such that for all $x, y \in X$:
$$\|T(x) - T(y)\| \le L \|x - y\|$$

---

**2.**
To find the fixed point, we substitute $x^\star = T(x^\star)$:
$$x^\star = \alpha x^\star + (1 - \alpha)c \implies (1 - \alpha)x^\star = (1 - \alpha)c$$
Since $0 < \alpha < 1$, the term $(1-\alpha) \neq 0$, allowing us to divide out to find the unique fixed point:
$$x^\star = c$$

To show geometric convergence, we evaluate the error propagation at step $k+1$:
$$x_{k+1} - x^\star = \left(\alpha x_k + (1 - \alpha)c\right) - c = \alpha x_k - \alpha c = \alpha(x_k - x^\star)$$
Taking the norm of both sides yields:
$$\|x_{k+1} - x^\star\| = \alpha \|x_k - x^\star\|$$
Applying this relationship inductively back to the initial guess $x_0$:
$$\|x_k - x^\star\| = \alpha^k \|x_0 - x^\star\|$$
Because $0 < \alpha < 1$, the error shrinks toward zero at a **geometric rate**, where the exact rate of convergence is the scalar **$\alpha$**.

---
**3.**
* **Fastest Convergence:** **PnP-ADMM (Section 6)** converges the fastest. The dual variable accumulation allows it to rapidly drive the system into agreement compared to the memoryless Picard loop or the simple averaging heuristic.
* **Highest PSNR:** **PnP-ADMM** achieves the highest and most stable final PSNR. Picard fixed-point iteration performs reasonably well but is highly sensitive to the scaling parameter $\rho$ and can stall or oscillate. Simple averaging lacks a robust optimization framework, often resulting in premature stalling or over-smoothed artifacts that yield the lowest PSNR of the three.
---
**4.**
At convergence ($x^k \to x^\star, v^k \to x^\star$), the optimality conditions of ADMM dictate that the scaled dual variable converges to the negative gradient of the data-fidelity term:
$$\rho u^\star = -\nabla f(x^\star)$$
* **Limiting Value Meaning:** The value $\rho u^\star$ represents the **equilibrium force** or directional tension required to bridge the gap between data consistency and the denoiser prior. It tells us exactly how much the data-space representation needs to be shifted to lie natively on the denoiser's image manifold.
* **Information Lost:** By dropping $u$, we eliminate the algorithm's "integral memory." Without it, the system cannot accumulate and rectify persistent discrepancies between the data step and the denoiser, losing the mathematical forcing mechanism that guarantees the primal residual $\|x - v\|_2$ is driven strictly to zero.
---
**5.**
Standard Picard iteration is entirely memoryless, calculating $x_{k+1}$ based solely on $x_k$. When dealing with complex image denoisers where the operator $T$ is non-expansive but not a strict contraction ($L \approx 1$), Picard iteration severely bottlenecks, causing slow linear convergence or cyclic oscillations. 

**Anderson Acceleration** mitigates this by maintaining a history of the last $m$ iterates. It dynamically computes a set of weights to construct a consensus profile from those past steps that actively minimizes the multi-step fixed-point residual $\|T(x) - x\|$. By taking its next step along this optimized vector rather than blindly executing the single-step operator, it acts like a non-linear momentum method—extrapolating past trends to bypass bottlenecks and dramatically accelerate convergence.



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6), dpi=100)

# PnP-ADMM (Section 6) - dynamically scales to its own length
plt.plot(range(1, len(history["psnr"]) + 1), history["psnr"], marker='o', linestyle='-', color='tab:green', linewidth=2.5, label='Section 6: PnP-ADMM')

# Consensus by Averaging (Section 10) - dynamically scales to 40 items
plt.plot(range(1, len(psnr_history_avg) + 1), psnr_history_avg, marker='s', linestyle='--', color='tab:blue', linewidth=2, label='Section 10: Averaging Consensus')

# Picard Fixed-Point (Section 11) - dynamically scales to its own length
plt.plot(range(1, len(psnr_history_picard) + 1), psnr_history_picard, marker='^', linestyle='-.', color='tab:red', linewidth=2, label='Section 11: Picard Fixed-Point')

plt.title("PnP Solver Comparison: PSNR vs. Iteration", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Iteration", fontsize=12)
plt.ylabel("PSNR (dB)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.7)
plt.legend(fontsize=11, loc='lower right', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

## 12. Gestalt and the role of priors

In psychology, **Gestalt principles** describe rules humans use to organize visual input into a whole:

* **Closure** — we fill in gaps and see complete shapes.
* **Continuity** — we follow smooth contours.
* **Proximity** — we group nearby elements together.

A strong prior $h(x)$ (or denoiser $D$) acts as an **algorithmic Gestalt**: it carries a model of "what natural images look like" and *enforces* those rules.

### Benefit

When the data is mildly corrupted, the prior cleans up the noise and produces sharp, plausible images.

### Risk: hallucination

When the data is **severely** corrupted (heavy blur, missing pixels, very low SNR), the data term $f$ is weak and contributes almost no information. The prior takes over and starts to **invent** details from its internal "what an image looks like" model. The result *looks* plausible but is not faithful to the actual scene.

### What you will do

1. Destroy the image with a **31×31** averaging blur, so $f$ contains essentially no high-frequency information.
2. Run an iterative loop that applies an aggressive denoiser **only** (no data term):
   $$x_{k+1} = D(x_k).$$
3. Watch the denoiser "dream up" structure. Measure PSNR — you will find that it doesn't really improve, because the hallucinated structure is not the true structure.


In [ ]:
# =============================================================
# Section 12 — Hallucination: a denoiser-only fixed-point iteration
# =============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d
from skimage.restoration import denoise_nl_means
from skimage.metrics import peak_signal_noise_ratio as psnr

# --- 1. Build a very strong blur kernel ----------------------------------
# TODO: kernel_size = 31. Build `strong_psf`, a uniform kernel that sums to 1
#       (each entry = 1 / (kernel_size**2)).
kernel_size = 31
strong_psf = np.ones((kernel_size, kernel_size)) / (kernel_size**2)

# --- 2. Obliterate the image --------------------------------------------
# TODO: strong_blur = convolve2d(img, strong_psf, mode='same', boundary='wrap')
# TODO: noisy_sb = strong_blur + Gaussian noise of std 0.02, clipped to [0, 1].
strong_blur = convolve2d(img, strong_psf, mode='same', boundary='wrap')
np.random.seed(42)
noise = np.random.normal(0, 0.02, size=img.shape)
noisy_sb = np.clip(strong_blur + noise, 0, 1)

# --- 3. Iterative denoiser-only loop (NOT PnP — no data term) -----------
# TODO: out = noisy_sb.copy(). Use an aggressive NLM strength (try h=0.12).
out = noisy_sb.copy() # TODO
h_strength = 0.06 # TODO
num_iters = 30

history_psnr = []

for k in range(num_iters):
    # TODO: out = denoise_nl_means(out.astype(np.float64), patch_size=7,
    #             patch_distance=10, h=h_strength, fast_mode=True)
    # TODO: out = np.clip(out, 0, 1)
    out = denoise_nl_means(out.astype(np.float64), patch_size=7, patch_distance=10, h=h_strength, fast_mode=True, channel_axis=None)
    out = np.clip(out, 0, 1)
    
    current_psnr = psnr(img, out, data_range=1.0)
    history_psnr.append(current_psnr)
    
    if (k + 1) % 5 == 0 or k == num_iters - 1:
        print(f"Iteration {k+1:02d}/{num_iters} | Current PSNR: {current_psnr:.2f} dB")

# --- 4. PSNR + plots -----------------------------------------------------
# TODO: psnr_noisy        = psnr(img, noisy_sb)
psnr_noisy = psnr(img, noisy_sb, data_range=1.0)

# TODO: psnr_hallucinated = psnr(img, out)
psnr_hallucinated = psnr(img, out, data_range=1.0)

# TODO: Plot a 1x3 row of the original `img`, the obliterated `noisy_sb`,
#       and the final `out`. Put each PSNR in the title.
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(img, cmap='gray')
ax[0].set_title('Original img')
ax[0].axis('off')

ax[1].imshow(noisy_sb, cmap='gray')
ax[1].set_title(f'Obliterated noisy_sb\nPSNR: {psnr_noisy:.2f} dB')
ax[1].axis('off')

ax[2].imshow(out, cmap='gray')
ax[2].set_title(f'Hallucinated out\nPSNR: {psnr_hallucinated:.2f} dB')
ax[2].axis('off')

plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(range(1, num_iters + 1), history_psnr, color='green', marker='o', markersize=3)
plt.title('PSNR vs. Iteration (Pure Denoiser)')
plt.xlabel('Iteration')
plt.ylabel('PSNR (dB)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# TODO: Print a short comment comparing psnr_noisy and psnr_hallucinated:
#       is the hallucinated image actually more "correct" than the blurry one?


print(f"PSNR of Noisy Input (noisy_sb): {psnr_noisy:.2f} dB")
print(f"PSNR of Hallucinated Output (out): {psnr_hallucinated:.2f} dB")
print("No, the hallucinated image is not more correct, despite looking sharper")
print("and cleaner to the human eye. Because the data fidelity term was entirely")
print("dropped, the loop lacked any anchor to the physical observation. The denoiser")
print("simply enforced its own internal architectural Gestalt rules clustering random noise into artificial,")
print("structures. It literally 'dreamed up' smooth regions and edges")
print("that do not exist in the true scene, which is why the final PSNR drops.")


### Section 12 — Questions

1. Look at the hallucinated image. In your own words, what kind of details has the denoiser "invented"? Be specific (e.g. "the eyes look sharp, but they are not in the same place as in the original").
2. Compare PSNR(noisy_sb) and PSNR(hallucinated). Even if the hallucinated image *looks* sharper, the PSNR often doesn't improve. Explain in 2–3 sentences why **PSNR can miss the perceived quality** in this regime.
3. In which **real-world** applications would hallucination be (a) acceptable / desirable, and (b) dangerous? Give one concrete example for each (e.g. photo enhancement vs. medical imaging).
4. **Design a test for hallucination** in a PnP reconstruction.
   * Hint: re-run the algorithm with two different but similarly strong priors. If they disagree in regions of low data fidelity, those regions are likely hallucinated.
   * Spell out the protocol: what would you compute, what would you visualize, what threshold would flag a region as "uncertain"?
5. The Gestalt analogy says the denoiser is an "algorithmic visual grammar". What grammatical rules has your NLM denoiser **internalized**, and how would those rules differ for a CNN trained on **medical** images? Speculate briefly.

*Your answers:*


**1.**

Instead of sharpening features, the Non-Local Means (NLM) denoiser has completely melted away every distinct human and structural element in the image. The astronaut's face, her smile, eyes, hair, and the complex metallic ring of the spacesuit helmet are entirely gone. In their place, the denoiser has "invented" a massive, ultra-smooth, abstract dark blob in the center of the frame surrounded by large, cloud-like gradients of white and grey in the upper corners. It took localized noise remnants and clustered them into a simplified, smooth silhouette that looks like minimalist graphic art, completely disconnected from the original scene.

---

**2.**

Output shows a drop from $17.08\text{ dB}$ down to $15.7\text{ dB}$. PSNR relies strictly on pixel-by-pixel Mean Squared Error ($MSE$). While the blurry image looks visually worse to us, its pixel values hover uniformly near the local average of the original image, keeping error distances modest. The hallucinated image creates vast regions of high-contrast, polarized dark and light tones; because these smooth abstract shapes completely misalign with the sharp structural layout of the true portrait, the localized pixel differences destroy the PSNR score.

---

**3.**

* **(a) Acceptable / Desirable:** *Generative Art, Special Effects, and Creative Upscaling.* In digital art synthesis or video game texture upscaling, absolute physical fidelity does not matter. The goal is removing low-quality compression noise and generating aesthetically smooth, high-contrast, visually pleasing contours, making this kind of radical simplification acceptable.
* **(b) Dangerous:** *Satellite Reconnaissance and Surveillance.* If an algorithm processing a heavily blurred satellite feed melts small, low-contrast structural targets (like vehicles, buildings, or runways) into massive, smooth, abstract terrain gradients or invents non-existent landscape borders, it will lead to catastrophic failures in intelligence gathering and tactical mapping.

---

**4.**

Real image features are anchored firmly by the physical data term $y$, whereas hallucinated structural elements are artifacts generated purely by the specific internal rules of the chosen prior.
* **Protocol Step 1 (Computation):** Run optimization setup twice under identical conditions, but swap out the denoiser. Run version $\hat{x}_A$ using the patch-matching NLM denoiser, and version $\hat{x}_B$ using a totally different architecture, like a deep-learning model (DnCNN) or a wavelet-based approach (BM3D). Compute a spatial difference map:
$$\Delta(i, j) = |\hat{x}_A(i, j) - \hat{x}_B(i, j)|$$
* **Protocol Step 2 (Visualization):** Plot a spatial error heatmap of $\Delta(i, j)$ overlaying the reconstruction. Areas showing intense variations reveal where the data term has lost control, leaving the two priors to "dream up" completely different structures.
* **Protocol Step 3 (Threshold Flag):** Set a dynamic tolerance threshold based on the baseline observation noise floor (e.g., $\tau = 1.5 \times \sigma_{\text{noise}}$). Any spatial region where $\Delta(i, j) > \tau$ is explicitly flagged with a bright overlay mask as a "High-Risk Hallucination Zone."

---

**5.**

The NLM denoiser utilizes a patch-redundancy grammar based entirely on spatial self-similarity. Its strict internal rule is: *"Every localized neighborhood must look like another neighborhood somewhere else in this image."* Deprived of a data anchor, it iteratively averages out distinct variations, collapsing the image into massive, uniform, low-frequency consensus regions (the smooth black blob and white gradients). 

Conversely, a CNN trained on specialized medical imaging (like brain MRIs) internalizes a strict semantic, anatomical grammar. Its hidden layers are hardwired with architectural rules like: *"Vessels form branching tubular networks"* or *"Cortex tissues form continuous, folding geometric ribbons."* If a medical CNN were run in a pure hallucination loop without data constraints, it wouldn't create a smooth, abstract blob like NLM; it would manufacture highly detailed, crisp, and incredibly convincing—but completely fake—anatomical folds , creating a massive risk of diagnostic deception.


## Submission

Before you submit, please:

1. **Restart the kernel and Run All.** The notebook must execute top to bottom without errors.
2. **Keep all outputs visible** (plots and printouts). We grade what we see; we do not re-run your notebook unless something looks wrong.
3. **Save the trained weights file** `tiny_dncnn_demo.pth` next to your notebook, so Section 7's evaluation cell can load it.
4. **Sanity-check that every TODO is replaced.** If a `raise NotImplementedError` or `None  # TODO` is still in your notebook, that part is not done.
5. **Answer every Questions cell.** Yes/No answers will not be graded. Use full sentences.

### Bonus (optional)

Pick **one** of the following extensions and add a new section at the bottom of the notebook:

* **B1.** Replace NLM with your trained `TinyDnCNN` in the Section 6 PnP-ADMM loop. Report the PSNR-vs-iteration curve. Did it help?
* **B2.** Run the Section 6 PnP-ADMM loop with **three** denoiser strengths inside the loop. Plot the three PSNR curves on one figure and write 3–4 sentences of analysis.
* **B3.** Implement a *simple* form of Anderson acceleration for the Section 11 Picard iteration (use `m=3` previous iterates). Compare convergence speed against plain Picard.

State which bonus you chose at the top of your bonus section.

---

Good luck. If you get stuck, re-read the markdown cell directly above the code cell — every TODO has the math or recipe spelled out there.


**Bonus: B2**

In [ ]:
# Bonus: B2

import numpy as np
import matplotlib.pyplot as plt
from skimage.restoration import denoise_nl_means
from skimage.metrics import peak_signal_noise_ratio as psnr

h_values = [0.03, 0.08, 0.20]
labels = ['Weak Denoiser (h=0.03)', 'Moderate Denoiser (h=0.08)', 'Strong Denoiser (h=0.20)']
colors = ['royalblue', 'forestgreen', 'darkorange']

all_psnr_histories = {}

for h_val, label in zip(h_values, labels):

    x = y.copy()
    v = x.copy()
    u = np.zeros_like(x)
    
    psnr_history = []
    
    for k in range(max_iters):

        z = v - u
        x = x_update_fft(y, H, rho, z)
        
        v = denoise_nl_means((x + u).astype(np.float64), patch_size=5, patch_distance=6, h=h_val, fast_mode=True)
        
        u = u + x - v

        current_psnr = psnr(img, np.clip(x, 0, 1))
        psnr_history.append(current_psnr)
        
    all_psnr_histories[label] = psnr_history

plt.figure(figsize=(9, 5))
for (label, history), color in zip(all_psnr_histories.items(), colors):
    plt.plot(history, label=label, color=color, linewidth=2)

plt.title('PnP-ADMM Convergence: Effect of Denoiser Strength ($h$)', fontsize=14)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('PSNR (dB)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

Too Weak **(h = 0.03)**: When the denoiser strength is too low, the algorithm fails to suppress the noise and blur artifacts introduced by the forward model, resulting in an unstable and under-regularized reconstruction that plateaus at a suboptimal PSNR.

Optimal **(h = 0.08)**: The moderate strength provides the ideal balance between the data fidelity term and the image prior, allowing the PSNR to climb steadily and achieve the highest overall reconstruction quality.

Too Strong **(h = 0.20)**: An overly aggressive denoiser aggressively wipes out structural details and edges alongside the noise during early iterations; this over-smoothing causes the PSNR to plateau prematurely at a lower value because fine details are permanently lost.